In [4]:

import os
import gc
import json
import random
import warnings
import numpy as np
import pandas as pd
import torch

# Reproducibility
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory
            / (1024 ** 3),
            2
        ),
        "GB"
    )
else:
    print("WARNING: GPU is not available.")


PyTorch: 2.14.1+cpu
CUDA available: False


In [5]:

%pip install -q transformers accelerate bitsandbytes peft trl datasets sentencepiece


In [6]:

import transformers
import accelerate
import bitsandbytes
import peft
import datasets

print("Transformers:", transformers.__version__)
print("Accelerate:", accelerate.__version__)
print("BitsAndBytes:", bitsandbytes.__version__)
print("PEFT:", peft.__version__)
print("Datasets:", datasets.__version__)
print("PyTorch:", torch.__version__)

print("\nCUDA available:", torch.cuda.is_available())


Transformers: 5.22.1
Accelerate: 1.15.0
BitsAndBytes: 0.50.2
PEFT: 0.21.2
Datasets: 4.8.5
PyTorch: 2.14.1+cpu

CUDA available: False


In [7]:

from google.colab import drive

drive.mount("/content/drive")


Mounted at /content/drive


In [8]:

PROJECT_ROOT = "/content/drive/MyDrive/BharatDocAI"

ANNOTATION_PATH = os.path.join(
    PROJECT_ROOT,
    "data/annotations/synthetic_documents.jsonl"
)

UNIFIED_DATASET_PATH = os.path.join(
    PROJECT_ROOT,
    "experiments/baselines/llm_extraction/unified_extraction_dataset.jsonl"
)

OUTPUT_DIR = os.path.join(
    PROJECT_ROOT,
    "experiments/finetuning"
)

METRICS_DIR = os.path.join(
    PROJECT_ROOT,
    "outputs/metrics/finetuning"
)

CHECKPOINT_DIR = os.path.join(
    PROJECT_ROOT,
    "models/checkpoints/qwen3_lora"
)

for path in [OUTPUT_DIR, METRICS_DIR, CHECKPOINT_DIR]:
    os.makedirs(path, exist_ok=True)

print("Project root:", PROJECT_ROOT)

for path in [ANNOTATION_PATH, UNIFIED_DATASET_PATH]:
    print(
        os.path.basename(path),
        "exists:",
        os.path.exists(path)
    )


Project root: /content/drive/MyDrive/BharatDocAI
synthetic_documents.jsonl exists: True
unified_extraction_dataset.jsonl exists: True


In [9]:

def load_jsonl(path):
    records = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))

    return records


annotations = load_jsonl(ANNOTATION_PATH)
unified_records = load_jsonl(UNIFIED_DATASET_PATH)

print("Original annotations:", len(annotations))
print("Unified OCR records:", len(unified_records))

print("\nUnified record keys:")
print(unified_records[0].keys())

print("\nSample OCR record:")
print(json.dumps(
    unified_records[0],
    ensure_ascii=False,
    indent=2
)[:2000])


Original annotations: 90
Unified OCR records: 4320

Unified record keys:
dict_keys(['document_id', 'language', 'category', 'template_family', 'split', 'ocr_engine', 'corruption', 'severity', 'ocr_text', 'ground_truth', 'ocr_status'])

Sample OCR record:
{
  "document_id": "english_invoice_001",
  "language": "english",
  "category": "invoice",
  "template_family": "T1",
  "split": "train",
  "ocr_engine": "tesseract",
  "corruption": "clean",
  "severity": "none",
  "ocr_text": "INVOICE\n\nCustomer Name:\n\nInvoice Date:\n\nInvoice Number:\n\nAddress:\n\nTotal Amount:\n\nKiran Rao\n\n01/01/2026\n\nINV-0001\n\nBengaluru\n\n1637",
  "ground_truth": {
    "name": "Kiran Rao",
    "date": "01/01/2026",
    "document_id": "INV-0001",
    "address": "Bengaluru",
    "amount": "1637"
  },
  "ocr_status": "success"
}


In [10]:

unified_df = pd.DataFrame(unified_records)

print("Total OCR records:", len(unified_df))

print("\nRecords by split:")
print(unified_df["split"].value_counts())

print("\nUnique documents by split:")
print(
    unified_df.groupby("split")["document_id"].nunique()
)

print("\nRecords by OCR engine:")
print(unified_df["ocr_engine"].value_counts())

print("\nRecords by language:")
print(unified_df["language"].value_counts())

print("\nCorruption types:")
print(unified_df["corruption"].value_counts())


Total OCR records: 4320

Records by split:
split
train         2592
validation     864
test           864
Name: count, dtype: int64

Unique documents by split:
split
test          18
train         54
validation    18
Name: document_id, dtype: int64

Records by OCR engine:
ocr_engine
tesseract    1440
easyocr      1440
paddleocr    1440
Name: count, dtype: int64

Records by language:
language
english    1440
hindi      1440
kannada    1440
Name: count, dtype: int64

Corruption types:
corruption
gaussian_blur       810
gaussian_noise      810
low_contrast        810
rotation            810
jpeg_compression    810
clean               270
Name: count, dtype: int64


In [11]:

clean_records = [
    r for r in unified_records
    if r["corruption"] == "clean"
    and r["ocr_status"] == "success"
]

train_records = [
    r for r in clean_records
    if r["split"] == "train"
]

validation_records = [
    r for r in clean_records
    if r["split"] == "validation"
]

test_records = [
    r for r in clean_records
    if r["split"] == "test"
]

print("Training OCR records:", len(train_records))
print("Validation OCR records:", len(validation_records))
print("Test OCR records:", len(test_records))

print("\nUnique training documents:",
      len(set(r["document_id"] for r in train_records)))

print("Unique validation documents:",
      len(set(r["document_id"] for r in validation_records)))

print("Unique test documents:",
      len(set(r["document_id"] for r in test_records)))


Training OCR records: 162
Validation OCR records: 54
Test OCR records: 54

Unique training documents: 54
Unique validation documents: 18
Unique test documents: 18


In [12]:

import re
import unicodedata

CATEGORY_SCHEMAS = {
    "invoice": [
        "name",
        "date",
        "document_id",
        "address",
        "amount"
    ],
    "application": [
        "name",
        "date",
        "document_id",
        "address",
        "phone"
    ],
    "certificate": [
        "name",
        "date",
        "document_id",
        "institution",
        "grade"
    ]
}


def normalize_field_value(value, field):
    value = unicodedata.normalize("NFC", str(value))

    value = value.replace("\u200c", "")
    value = value.replace("\u200d", "")

    value = value.strip().casefold()

    if field in ["date", "phone", "amount", "document_id"]:
        value = re.sub(r"[\s/\\\-:.,|]", "", value)
    else:
        value = re.sub(r"\s+", " ", value)

    return value


def is_field_supported(expected, ocr_text, field):

    expected_norm = normalize_field_value(expected, field)
    ocr_norm = normalize_field_value(ocr_text, field)

    return (
        expected_norm != ""
        and expected_norm in ocr_norm
    )


print("Category schemas loaded.")

for category, fields in CATEGORY_SCHEMAS.items():
    print(category, ":", fields)


Category schemas loaded.
invoice : ['name', 'date', 'document_id', 'address', 'amount']
application : ['name', 'date', 'document_id', 'address', 'phone']
certificate : ['name', 'date', 'document_id', 'institution', 'grade']


In [13]:

def build_training_example(record):

    category = record["category"]
    language = record["language"]

    expected_fields = CATEGORY_SCHEMAS[category]
    ground_truth = record["ground_truth"]
    ocr_text = record["ocr_text"]

    target = {}

    for field in expected_fields:

        expected_value = ground_truth.get(field, "")

        supported = is_field_supported(
            expected_value,
            ocr_text,
            field
        )

        target[field] = (
            str(expected_value) if supported else ""
        )

    return {
        "document_id": record["document_id"],
        "language": language,
        "category": category,
        "ocr_engine": record["ocr_engine"],
        "input_text": (
            f"You are an information extraction system.\n\n"
            f"Document category: {category}\n"
            f"Document language: {language}\n\n"
            f"Extract these fields: "
            f"{', '.join(expected_fields)}.\n"
            f"Return exactly these fields as valid JSON.\n"
            f"Extract only information supported by the OCR text.\n"
            f"If a field is missing or unreadable, return an empty string.\n\n"
            f"OCR TEXT:\n{ocr_text}"
        ),
        "target_json": json.dumps(
            target,
            ensure_ascii=False,
            indent=2
        ),
        "target": target
    }


train_examples = [
    build_training_example(record)
    for record in train_records
]

print("Training examples created:", len(train_examples))

print("\nSample training example:")
print(train_examples[0]["input_text"])

print("\nExpected JSON:")
print(train_examples[0]["target_json"])


Training examples created: 162

Sample training example:
You are an information extraction system.

Document category: invoice
Document language: english

Extract these fields: name, date, document_id, address, amount.
Return exactly these fields as valid JSON.
Extract only information supported by the OCR text.
If a field is missing or unreadable, return an empty string.

OCR TEXT:
INVOICE

Customer Name:

Invoice Date:

Invoice Number:

Address:

Total Amount:

Kiran Rao

01/01/2026

INV-0001

Bengaluru

1637

Expected JSON:
{
  "name": "Kiran Rao",
  "date": "01/01/2026",
  "document_id": "INV-0001",
  "address": "Bengaluru",
  "amount": "1637"
}


In [14]:

support_rows = []

for record, example in zip(train_records, train_examples):

    for field in CATEGORY_SCHEMAS[record["category"]]:

        support_rows.append({
            "document_id": record["document_id"],
            "language": record["language"],
            "category": record["category"],
            "ocr_engine": record["ocr_engine"],
            "field": field,
            "supported": example["target"][field] != ""
        })

support_df = pd.DataFrame(support_rows)

print("Total training fields:", len(support_df))

print(
    "Supported fields:",
    support_df["supported"].sum()
)

print(
    "Unsupported fields:",
    (~support_df["supported"]).sum()
)

print(
    "Overall OCR field support:",
    round(support_df["supported"].mean() * 100, 2),
    "%"
)

print("\nSupport by language:")
print(
    support_df.groupby("language")["supported"]
    .mean().mul(100).round(2)
)

print("\nSupport by OCR engine:")
print(
    support_df.groupby("ocr_engine")["supported"]
    .mean().mul(100).round(2)
)

print("\nSupport by field:")
print(
    support_df.groupby("field")["supported"]
    .mean().mul(100).round(2)
)


Total training fields: 810
Supported fields: 621
Unsupported fields: 189
Overall OCR field support: 76.67 %

Support by language:
language
english    99.26
hindi      75.93
kannada    54.81
Name: supported, dtype: float64

Support by OCR engine:
ocr_engine
easyocr      86.67
paddleocr    67.78
tesseract    75.56
Name: supported, dtype: float64

Support by field:
field
address        86.11
amount         90.74
date           83.33
document_id    58.02
grade          57.41
institution    79.63
name           80.86
phone          83.33
Name: supported, dtype: float64


In [15]:

missing_examples = []

for example in train_examples:

    missing_fields = [
        field
        for field, value in example["target"].items()
        if value == ""
    ]

    if missing_fields:
        missing_examples.append({
            "document_id": example["document_id"],
            "language": example["language"],
            "category": example["category"],
            "ocr_engine": example["ocr_engine"],
            "missing_fields": missing_fields
        })

missing_df = pd.DataFrame(missing_examples)

print("Examples with at least one missing field:",
      len(missing_df))

display(missing_df.head(20))


Examples with at least one missing field: 91


,document_id,language,category,ocr_engine,missing_fields
0,english_certificate_022,english,certificate,tesseract,[grade]
1,english_certificate_023,english,certificate,tesseract,[grade]
2,hindi_invoice_031,hindi,invoice,tesseract,"[date, document_id]"
3,hindi_invoice_032,hindi,invoice,tesseract,"[date, document_id]"
4,hindi_invoice_033,hindi,invoice,tesseract,"[document_id, amount]"
5,hindi_invoice_034,hindi,invoice,tesseract,"[document_id, amount]"
6,hindi_invoice_035,hindi,invoice,tesseract,[document_id]
7,hindi_invoice_036,hindi,invoice,tesseract,[document_id]
8,hindi_application_041,hindi,application,tesseract,"[date, document_id, phone]"
9,hindi_application_042,hindi,application,tesseract,"[date, document_id]"


In [16]:
# Inspect examples with unsupported fields

for example in train_examples:
    if example["missing_fields"] if "missing_fields" in example else False:
        print(example)
        break

In [17]:
# Find examples containing empty target fields

for example in train_examples:
    target = json.loads(example["target_json"])

    if any(value == "" for value in target.values()):
        print("INPUT:")
        print(example["input_text"])

        print("\nTARGET:")
        print(json.dumps(target, ensure_ascii=False, indent=2))

        print("\n" + "=" * 60)
        break

INPUT:
You are an information extraction system.

Document category: certificate
Document language: english

Extract these fields: name, date, document_id, institution, grade.
Return exactly these fields as valid JSON.
Extract only information supported by the OCR text.
If a field is missing or unreadable, return an empty string.

OCR TEXT:
CERTIFICATE

Recipient Name:

Issue Date:

Certificate Number:

Institution:

Grade:

Arjun Sharma

22/01/2026

CERT-0022

Government Science College

TARGET:
{
  "name": "Arjun Sharma",
  "date": "22/01/2026",
  "document_id": "CERT-0022",
  "institution": "Government Science College",
  "grade": ""
}



In [18]:
# Cell 13: Prepare Qwen3 chat-format training data

SYSTEM_PROMPT = """
You are an information extraction system.
Extract only information explicitly supported by the OCR text.
Never guess or invent missing values.
Return exactly the requested fields as valid JSON.
Use an empty string when a field is missing or unreadable.
"""

def create_conversation(example):

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT.strip()
        },
        {
            "role": "user",
            "content": example["input_text"]
        },
        {
            "role": "assistant",
            "content": example["target_json"]
        }
    ]

    return {
        "messages": messages
    }


train_conversations = [
    create_conversation(example)
    for example in train_examples
]

print("Training conversations:", len(train_conversations))

print("\nSample conversation:")
for message in train_conversations[0]["messages"]:
    print(f"\n{message['role'].upper()}:")
    print(message["content"])

Training conversations: 162

Sample conversation:

SYSTEM:
You are an information extraction system.
Extract only information explicitly supported by the OCR text.
Never guess or invent missing values.
Return exactly the requested fields as valid JSON.
Use an empty string when a field is missing or unreadable.

USER:
You are an information extraction system.

Document category: invoice
Document language: english

Extract these fields: name, date, document_id, address, amount.
Return exactly these fields as valid JSON.
Extract only information supported by the OCR text.
If a field is missing or unreadable, return an empty string.

OCR TEXT:
INVOICE

Customer Name:

Invoice Date:

Invoice Number:

Address:

Total Amount:

Kiran Rao

01/01/2026

INV-0001

Bengaluru

1637

ASSISTANT:
{
  "name": "Kiran Rao",
  "date": "01/01/2026",
  "document_id": "INV-0001",
  "address": "Bengaluru",
  "amount": "1637"
}


In [19]:
# Cell 14: Load Qwen3 tokenizer

from transformers import AutoTokenizer

MODEL_NAME = "Qwen/Qwen3-4B"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer loaded successfully")
print("Vocabulary size:", len(tokenizer))
print("Pad token:", tokenizer.pad_token)
print("EOS token:", tokenizer.eos_token)

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

Tokenizer loaded successfully
Vocabulary size: 151669
Pad token: <|endoftext|>
EOS token: <|im_end|>


In [20]:
# Cell 15: Tokenize conversations with assistant-only loss

MAX_LENGTH = 1024

def tokenize_conversation(conversation):

    messages = conversation["messages"]

    # Format everything before the assistant response
    prompt_messages = messages[:-1]

    prompt_text = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    # Format the complete conversation
    full_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
        enable_thinking=False
    )

    # Tokenize prompt and complete conversation
    prompt_ids = tokenizer(
        prompt_text,
        add_special_tokens=False
    )["input_ids"]

    full_ids = tokenizer(
        full_text,
        add_special_tokens=False
    )["input_ids"]

    # Truncate if necessary
    full_ids = full_ids[:MAX_LENGTH]
    prompt_length = min(len(prompt_ids), len(full_ids))

    # Mask prompt tokens with -100
    labels = [-100] * prompt_length + full_ids[prompt_length:]

    return {
        "input_ids": full_ids,
        "attention_mask": [1] * len(full_ids),
        "labels": labels
    }


tokenized_train = [
    tokenize_conversation(conversation)
    for conversation in train_conversations
]

print("Tokenization completed!")
print("Training samples:", len(tokenized_train))

print("\nFirst sample token count:",
      len(tokenized_train[0]["input_ids"]))

print("First sample supervised tokens:",
      sum(x != -100 for x in tokenized_train[0]["labels"]))

print("First sample masked tokens:",
      sum(x == -100 for x in tokenized_train[0]["labels"]))

Tokenization completed!
Training samples: 162

First sample token count: 239
First sample supervised tokens: 62
First sample masked tokens: 177


In [21]:
# Cell 16: Training data integrity checks

lengths = [len(x["input_ids"]) for x in tokenized_train]

supervised_counts = [
    sum(token != -100 for token in x["labels"])
    for x in tokenized_train
]

truncated = sum(length >= MAX_LENGTH for length in lengths)
empty_targets = sum(count == 0 for count in supervised_counts)

print("Total samples:", len(tokenized_train))
print("Maximum sequence length:", max(lengths))
print("Average sequence length:", round(np.mean(lengths), 2))
print("Truncated samples:", truncated)
print("Samples with empty supervised targets:", empty_targets)

assert len(tokenized_train) == 162
assert truncated == 0
assert empty_targets == 0

print("\nAll training data integrity checks passed!")

Total samples: 162
Maximum sequence length: 427
Average sequence length: 303.01
Truncated samples: 0
Samples with empty supervised targets: 0

All training data integrity checks passed!


In [22]:
# Cell 17: Load Qwen3-4B with 4-bit quantization

from transformers import (
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

from peft import (
    prepare_model_for_kbit_training,
    LoraConfig,
    get_peft_model
)

# 1. Configure 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# 2. Load the base model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

# 3. Disable cache for training
model.config.use_cache = False

# 4. Prepare model for QLoRA training
model = prepare_model_for_kbit_training(model)

print("Base model loaded successfully!")
print("Quantization:", model.is_quantized)
print("Model device:", model.get_input_embeddings().weight.device)
print("GPU memory allocated:",
      round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")

model.safetensors.index.json:   0%|          | 0.00/32.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Base model loaded successfully!
Quantization: True
Model device: cpu
GPU memory allocated: 0.0 GB


In [23]:
# Cell 18: Configure and attach LoRA adapters

model.gradient_checkpointing_enable()

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ]
)

model = get_peft_model(model, lora_config)

# Display trainable parameters
model.print_trainable_parameters()

trainable params: 16,515,072 || all params: 4,038,983,168 || trainable%: 0.4089


In [24]:
# Cell 19: Create PyTorch Dataset and DataLoader

from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

class BharatDocDataset(Dataset):

    def __init__(self, tokenized_data):
        self.data = tokenized_data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        item = self.data[index]

        return {
            "input_ids": torch.tensor(
                item["input_ids"], dtype=torch.long
            ),
            "attention_mask": torch.tensor(
                item["attention_mask"], dtype=torch.long
            ),
            "labels": torch.tensor(
                item["labels"], dtype=torch.long
            )
        }


def collate_fn(batch):

    input_ids = pad_sequence(
        [item["input_ids"] for item in batch],
        batch_first=True,
        padding_value=tokenizer.pad_token_id
    )

    attention_mask = pad_sequence(
        [item["attention_mask"] for item in batch],
        batch_first=True,
        padding_value=0
    )

    labels = pad_sequence(
        [item["labels"] for item in batch],
        batch_first=True,
        padding_value=-100
    )

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }


train_dataset = BharatDocDataset(tokenized_train)

print("PyTorch dataset created:", len(train_dataset))

PyTorch dataset created: 162


In [25]:
# Cell 20: Training configuration

from torch.optim import AdamW

EPOCHS = 5
BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 4

LEARNING_RATE = 2e-4
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 0.3

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn
)

optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

steps_per_epoch = len(train_loader)
optimizer_steps_per_epoch = (
    steps_per_epoch + GRADIENT_ACCUMULATION_STEPS - 1
) // GRADIENT_ACCUMULATION_STEPS

print("Training configuration")
print("----------------------")
print("Epochs:", EPOCHS)
print("Batch size:", BATCH_SIZE)
print("Gradient accumulation:", GRADIENT_ACCUMULATION_STEPS)
print("Effective batch size:",
      BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS)
print("Learning rate:", LEARNING_RATE)
print("Batches per epoch:", steps_per_epoch)
print("Optimizer steps per epoch:", optimizer_steps_per_epoch)
print("Total optimizer steps:",
      EPOCHS * optimizer_steps_per_epoch)

Training configuration
----------------------
Epochs: 5
Batch size: 2
Gradient accumulation: 4
Effective batch size: 8
Learning rate: 0.0002
Batches per epoch: 81
Optimizer steps per epoch: 21
Total optimizer steps: 105


In [26]:
# Cell 21: Prepare validation examples

validation_examples = []

for record in validation_records:

    category = record["category"]
    language = record["language"]
    ocr_text = record["ocr_text"]
    ground_truth = record["ground_truth"]

    fields = CATEGORY_SCHEMAS[category]

    input_text = f"""You are an information extraction system.

Document category: {category}
Document language: {language}

Extract these fields: {", ".join(fields)}.
Return exactly these fields as valid JSON.
Extract only information supported by the OCR text.
If a field is missing or unreadable, return an empty string.

OCR TEXT:
{ocr_text}
"""

    target = {}

    for field in fields:
        expected = ground_truth.get(field, "")

        if is_field_supported(expected, ocr_text, field):
            target[field] = expected
        else:
            target[field] = ""

    validation_examples.append({
        "document_id": record["document_id"],
        "language": language,
        "category": category,
        "ocr_engine": record["ocr_engine"],
        "input_text": input_text,
        "target_json": json.dumps(
            target,
            ensure_ascii=False,
            indent=2
        )
    })

print("Validation examples:", len(validation_examples))
print("Unique documents:", len(set(
    x["document_id"] for x in validation_examples
)))

print("\nSample validation target:")
print(validation_examples[0]["target_json"])

Validation examples: 54
Unique documents: 18

Sample validation target:
{
  "name": "Ravi Sharma",
  "date": "07/01/2026",
  "document_id": "INV-0007",
  "address": "Tumakuru",
  "amount": "2459"
}


In [27]:
# Cell 22: Tokenize validation data

validation_conversations = [
    create_conversation(example)
    for example in validation_examples
]

tokenized_validation = [
    tokenize_conversation(conversation)
    for conversation in validation_conversations
]

validation_dataset = BharatDocDataset(tokenized_validation)

print("Validation tokenization completed!")
print("Validation samples:", len(validation_dataset))

print(
    "Maximum validation sequence length:",
    max(len(x["input_ids"]) for x in tokenized_validation)
)

Validation tokenization completed!
Validation samples: 54
Maximum validation sequence length: 427


In [1]:
# Cell 23: QLoRA training loop with validation and best checkpoint saving

from transformers import get_linear_schedule_with_warmup

import math
import json
import os
import torch

# 1. Prepare output directory
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(METRICS_DIR, exist_ok=True)

# 2. Optimize only trainable LoRA parameters
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

total_steps = EPOCHS * optimizer_steps_per_epoch
warmup_steps = max(1, int(total_steps * 0.05))

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

# 3. Mixed precision gradient scaler
scaler = torch.amp.GradScaler("cuda")

device = model.get_input_embeddings().weight.device

best_validation_loss = float("inf")
best_epoch = None
training_history = []

print("Starting QLoRA training...")
print("Device:", device)
print("Total optimizer steps:", total_steps)
print("Warmup steps:", warmup_steps)

# 4. Training loop
for epoch in range(EPOCHS):

    model.train()
    optimizer.zero_grad(set_to_none=True)

    total_train_loss = 0.0
    train_examples_seen = 0

    for step, batch in enumerate(train_loader):

        batch = {
            key: value.to(device)
            for key, value in batch.items()
        }

        # Correctly scale the final incomplete accumulation group
        group_start = (
            step // GRADIENT_ACCUMULATION_STEPS
        ) * GRADIENT_ACCUMULATION_STEPS

        group_size = min(
            GRADIENT_ACCUMULATION_STEPS,
            len(train_loader) - group_start
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):
            outputs = model(**batch)
            loss = outputs.loss

        if not torch.isfinite(loss):
            raise RuntimeError(
                f"Non-finite training loss at epoch {epoch + 1}, "
                f"batch {step + 1}"
            )

        # Accumulate gradients
        scaler.scale(loss / group_size).backward()

        batch_size = batch["input_ids"].size(0)
        total_train_loss += loss.item() * batch_size
        train_examples_seen += batch_size

        # Optimizer update after each accumulation group
        is_accumulation_step = (
            (step + 1) % GRADIENT_ACCUMULATION_STEPS == 0
        )

        is_last_batch = (step + 1) == len(train_loader)

        if is_accumulation_step or is_last_batch:

            scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                MAX_GRAD_NORM
            )

            scaler.step(optimizer)
            scaler.update()

            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

    average_train_loss = (
        total_train_loss / train_examples_seen
    )

    # 5. Validation
    model.eval()

    total_validation_loss = 0.0
    validation_examples_seen = 0

    with torch.no_grad():

        for batch in DataLoader(
            validation_dataset,
            batch_size=BATCH_SIZE,
            shuffle=False,
            collate_fn=collate_fn
        ):

            batch = {
                key: value.to(device)
                for key, value in batch.items()
            }

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16
            ):
                outputs = model(**batch)

            val_loss = outputs.loss

            if not torch.isfinite(val_loss):
                raise RuntimeError(
                    f"Non-finite validation loss at epoch {epoch + 1}"
                )

            batch_size = batch["input_ids"].size(0)

            total_validation_loss += (
                val_loss.item() * batch_size
            )

            validation_examples_seen += batch_size

    average_validation_loss = (
        total_validation_loss / validation_examples_seen
    )

    # 6. Record results
    epoch_result = {
        "epoch": epoch + 1,
        "train_loss": average_train_loss,
        "validation_loss": average_validation_loss,
        "learning_rate": scheduler.get_last_lr()[0]
    }

    training_history.append(epoch_result)

    print(f"\nEpoch {epoch + 1}/{EPOCHS}")
    print(f"Training loss:   {average_train_loss:.4f}")
    print(f"Validation loss: {average_validation_loss:.4f}")
    print(f"Learning rate:   {scheduler.get_last_lr()[0]:.8f}")

    # 7. Save best checkpoint
    if average_validation_loss < best_validation_loss:

        best_validation_loss = average_validation_loss
        best_epoch = epoch + 1

        model.save_pretrained(CHECKPOINT_DIR)
        tokenizer.save_pretrained(CHECKPOINT_DIR)

        print("Best LoRA checkpoint saved!")

    # Save history after every epoch
    history_path = os.path.join(
        METRICS_DIR,
        "qwen3_lora_training_history.json"
    )

    with open(history_path, "w") as f:
        json.dump(training_history, f, indent=2)

    torch.cuda.empty_cache()


print("\nTraining completed!")
print("Best epoch:", best_epoch)
print("Best validation loss:", round(best_validation_loss, 4))
print("Checkpoint directory:", CHECKPOINT_DIR)

NameError: name 'CHECKPOINT_DIR' is not defined

In [ ]:
# Cell 24: Load the best LoRA checkpoint

model.load_adapter(
    CHECKPOINT_DIR,
    adapter_name="best"
)

model.set_adapter("best")
model.eval()

# Enable cache for inference
model.config.use_cache = True

print("Best checkpoint loaded!")
print("Active adapter:", model.active_adapter)
print("Best epoch:", best_epoch)
print("Best validation loss:", best_validation_loss)

torch.cuda.empty_cache()

In [ ]:
# Cell 25: Single-example inference test (corrected)

def generate_extraction(input_text):

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT.strip()
        },
        {
            "role": "user",
            "content": input_text
        }
    ]

    # Tokenize the conversation
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        enable_thinking=False,
        return_dict=True,
        return_tensors="pt"
    )

    device = model.get_input_embeddings().weight.device

    # Move all input tensors to GPU
    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    # Generate response
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=180,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    # Extract only newly generated tokens
    prompt_length = inputs["input_ids"].shape[-1]

    generated_tokens = outputs[0][prompt_length:]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return response.strip()


# Test one validation example
sample = validation_examples[0]

prediction = generate_extraction(sample["input_text"])

print("DOCUMENT:", sample["document_id"])

print("\nMODEL PREDICTION:")
print(prediction)

print("\nEXPECTED TARGET:")
print(sample["target_json"])

In [28]:

# Cell 26: Generate predictions for all validation examples

import json
import re
from tqdm.auto import tqdm

lora_validation_results = []

for example in tqdm(validation_examples):

    prediction_text = generate_extraction(example["input_text"])

    # Extract JSON safely
    try:
        start = prediction_text.find("{")
        end = prediction_text.rfind("}") + 1

        if start != -1 and end > start:
            prediction = json.loads(prediction_text[start:end])
            json_valid = True
        else:
            prediction = {}
            json_valid = False

    except json.JSONDecodeError:
        prediction = {}
        json_valid = False

    # Original ground truth, not support-filtered targets
    original_record = next(
        r for r in validation_records
        if r["document_id"] == example["document_id"]
        and r["ocr_engine"] == example["ocr_engine"]
    )

    ground_truth = original_record["ground_truth"]

    lora_validation_results.append({
        "document_id": example["document_id"],
        "language": example["language"],
        "category": example["category"],
        "ocr_engine": example["ocr_engine"],
        "prediction": prediction,
        "ground_truth": ground_truth,
        "json_valid": json_valid
    })

print("Total validation predictions:", len(lora_validation_results))
print("Valid JSON:", sum(r["json_valid"] for r in lora_validation_results))
print("Invalid JSON:", sum(not r["json_valid"] for r in lora_validation_results))

print("\nFirst prediction:")
print(json.dumps(lora_validation_results[0], indent=2, ensure_ascii=False))


  0%|          | 0/54 [00:00<?, ?it/s]

NameError: name 'generate_extraction' is not defined

In [ ]:

# Cell 27: Evaluate LoRA validation performance

import pandas as pd
import numpy as np

evaluation_rows = []

for record in lora_validation_results:

    fields = CATEGORY_SCHEMAS[record["category"]]

    prediction = record["prediction"]
    ground_truth = record["ground_truth"]

    correct_raw = 0
    correct_normalized = 0
    total_fields = len(fields)

    for field in fields:

        predicted_value = str(prediction.get(field, ""))
        expected_value = str(ground_truth.get(field, ""))

        # Raw comparison
        raw_match = predicted_value.strip() == expected_value.strip()

        # Normalized comparison
        normalized_match = (
            normalize_field_value(predicted_value, field)
            ==
            normalize_field_value(expected_value, field)
        )

        correct_raw += int(raw_match)
        correct_normalized += int(normalized_match)

    evaluation_rows.append({
        "document_id": record["document_id"],
        "language": record["language"],
        "category": record["category"],
        "ocr_engine": record["ocr_engine"],
        "correct_raw": correct_raw,
        "correct_normalized": correct_normalized,
        "total_fields": total_fields,
        "raw_accuracy": correct_raw / total_fields,
        "normalized_accuracy": correct_normalized / total_fields,
        "exact_match": correct_normalized == total_fields,
        "json_valid": record["json_valid"]
    })

lora_metrics_df = pd.DataFrame(evaluation_rows)

# Overall metrics
total_fields = lora_metrics_df["total_fields"].sum()

overall_metrics = {
    "Total records": len(lora_metrics_df),
    "Total fields": total_fields,
    "Valid JSON": int(lora_metrics_df["json_valid"].sum()),
    "Raw field accuracy": (
        lora_metrics_df["correct_raw"].sum() / total_fields * 100
    ),
    "Normalized field accuracy": (
        lora_metrics_df["correct_normalized"].sum() / total_fields * 100
    ),
    "Exact document match": (
        lora_metrics_df["exact_match"].mean() * 100
    )
}

print("OVERALL LORA VALIDATION RESULTS")
print("=" * 45)

for metric, value in overall_metrics.items():
    if isinstance(value, (float, np.floating)):
        print(f"{metric}: {value:.2f}%")
    else:
        print(f"{metric}: {value}")


# Language-wise metrics
language_metrics = (
    lora_metrics_df.groupby("language")
    .agg(
        documents=("document_id", "count"),
        correct_fields=("correct_normalized", "sum"),
        total_fields=("total_fields", "sum"),
        exact_match=("exact_match", "mean")
    )
)

language_metrics["field_accuracy"] = (
    language_metrics["correct_fields"]
    / language_metrics["total_fields"] * 100
)

language_metrics["exact_match"] *= 100

print("\nLANGUAGE-WISE PERFORMANCE")
display(language_metrics.round(2))


# OCR-engine-wise metrics
engine_metrics = (
    lora_metrics_df.groupby("ocr_engine")
    .agg(
        documents=("document_id", "count"),
        correct_fields=("correct_normalized", "sum"),
        total_fields=("total_fields", "sum"),
        exact_match=("exact_match", "mean")
    )
)

engine_metrics["field_accuracy"] = (
    engine_metrics["correct_fields"]
    / engine_metrics["total_fields"] * 100
)

engine_metrics["exact_match"] *= 100

print("\nOCR ENGINE-WISE PERFORMANCE")
display(engine_metrics.round(2))


# Save results
lora_metrics_df.to_csv(
    os.path.join(METRICS_DIR, "qwen3_lora_validation_metrics.csv"),
    index=False
)

print("\nMetrics saved successfully.")


In [ ]:

# Cell 28: Field-level LoRA error analysis

field_results = []

for record in lora_validation_results:

    fields = CATEGORY_SCHEMAS[record["category"]]

    for field in fields:

        predicted = str(record["prediction"].get(field, ""))
        expected = str(record["ground_truth"].get(field, ""))

        normalized_predicted = normalize_field_value(predicted, field)
        normalized_expected = normalize_field_value(expected, field)

        field_results.append({
            "document_id": record["document_id"],
            "language": record["language"],
            "category": record["category"],
            "ocr_engine": record["ocr_engine"],
            "field": field,
            "expected": expected,
            "predicted": predicted,
            "correct": normalized_predicted == normalized_expected,
            "prediction_empty": predicted.strip() == ""
        })

field_df = pd.DataFrame(field_results)

# Overall field performance
field_summary = (
    field_df.groupby("field")
    .agg(
        total=("correct", "count"),
        correct=("correct", "sum"),
        empty_predictions=("prediction_empty", "sum")
    )
)

field_summary["accuracy"] = (
    field_summary["correct"] / field_summary["total"] * 100
)

field_summary["empty_prediction_rate"] = (
    field_summary["empty_predictions"] / field_summary["total"] * 100
)

print("FIELD-WISE PERFORMANCE")
display(field_summary.round(2))


# Language and field breakdown
print("\nLANGUAGE-WISE FIELD PERFORMANCE")

language_field_summary = (
    field_df.groupby(["language", "field"])["correct"]
    .mean()
    .mul(100)
    .unstack()
)

display(language_field_summary.round(2))


# Inspect incorrect predictions
errors = field_df[field_df["correct"] == False]

print("\nTOTAL INCORRECT FIELDS:", len(errors))
print("\nSAMPLE ERRORS:")

display(errors[
    ["document_id", "language", "ocr_engine",
     "field", "expected", "predicted"]
].head(25))


# Save error analysis
field_df.to_csv(
    os.path.join(METRICS_DIR, "qwen3_lora_field_error_analysis.csv"),
    index=False
)

print("\nField-level analysis saved.")


In [ ]:

# Cell 29: Diagnose LoRA abstention against OCR evidence

diagnostic_rows = []

for result in lora_validation_results:

    # Retrieve the corresponding OCR record
    ocr_record = next(
        r for r in validation_records
        if r["document_id"] == result["document_id"]
        and r["ocr_engine"] == result["ocr_engine"]
    )

    fields = CATEGORY_SCHEMAS[result["category"]]

    for field in fields:

        expected = str(result["ground_truth"].get(field, ""))
        predicted = str(result["prediction"].get(field, ""))
        ocr_text = ocr_record["ocr_text"]

        # Check whether the expected value is present in OCR
        supported = is_field_supported(
            expected,
            ocr_text,
            field
        )

        prediction_empty = predicted.strip() == ""

        correct = (
            normalize_field_value(predicted, field)
            ==
            normalize_field_value(expected, field)
        )

        diagnostic_rows.append({
            "document_id": result["document_id"],
            "language": result["language"],
            "ocr_engine": result["ocr_engine"],
            "field": field,
            "ocr_supported": supported,
            "prediction_empty": prediction_empty,
            "correct": correct,
            "false_abstention": supported and prediction_empty,
            "unsupported_nonempty": not supported and not prediction_empty,
            "expected": expected,
            "predicted": predicted
        })

diagnostic_df = pd.DataFrame(diagnostic_rows)


# Overall diagnostic
print("OVERALL OCR SUPPORT AND ABSTENTION")
print("=" * 45)

print(
    "Expected values present in OCR:",
    diagnostic_df["ocr_supported"].sum()
)

print(
    "Empty predictions:",
    diagnostic_df["prediction_empty"].sum()
)

print(
    "False abstentions:",
    diagnostic_df["false_abstention"].sum()
)

print(
    "Non-empty predictions without exact OCR support:",
    diagnostic_df["unsupported_nonempty"].sum()
)


# Breakdown by OCR engine
print("\nOCR ENGINE-WISE DIAGNOSTIC")

engine_diagnostic = (
    diagnostic_df.groupby("ocr_engine")
    .agg(
        total_fields=("field", "count"),
        supported_fields=("ocr_supported", "sum"),
        empty_predictions=("prediction_empty", "sum"),
        false_abstentions=("false_abstention", "sum"),
        unsupported_nonempty=("unsupported_nonempty", "sum")
    )
)

display(engine_diagnostic)


# Breakdown by language
print("\nLANGUAGE-WISE DIAGNOSTIC")

language_diagnostic = (
    diagnostic_df.groupby("language")
    .agg(
        total_fields=("field", "count"),
        supported_fields=("ocr_supported", "sum"),
        empty_predictions=("prediction_empty", "sum"),
        false_abstentions=("false_abstention", "sum"),
        unsupported_nonempty=("unsupported_nonempty", "sum")
    )
)

display(language_diagnostic)


# Inspect cases where OCR contains the answer but LoRA abstained
print("\nFALSE ABSTENTION EXAMPLES")

false_abstention_examples = diagnostic_df[
    diagnostic_df["false_abstention"]
]

display(
    false_abstention_examples[
        ["document_id", "language", "ocr_engine",
         "field", "expected", "predicted"]
    ].head(30)
)


# Save diagnostic
diagnostic_df.to_csv(
    os.path.join(
        METRICS_DIR,
        "qwen3_lora_ocr_support_diagnostic.csv"
    ),
    index=False
)

print("\nDiagnostic saved successfully.")


In [ ]:

# Cell 30: Inspect all false abstention cases

false_abstention_ids = diagnostic_df[
    diagnostic_df["false_abstention"]
]

for _, row in false_abstention_ids.iterrows():

    record = next(
        r for r in validation_records
        if r["document_id"] == row["document_id"]
        and r["ocr_engine"] == row["ocr_engine"]
    )

    print("\n" + "=" * 75)
    print("DOCUMENT:", row["document_id"])
    print("LANGUAGE:", row["language"])
    print("OCR ENGINE:", row["ocr_engine"])
    print("FIELD:", row["field"])
    print("EXPECTED VALUE:", row["expected"])
    print("PREDICTED VALUE:", repr(row["predicted"]))

    print("\nOCR TEXT:")
    print(record["ocr_text"])

print("\nInspection completed.")


In [ ]:

# Cell 31: Inspect saved baseline prediction files

BASELINE_DIR = os.path.join(
    PROJECT_ROOT,
    "experiments/baselines/llm_extraction"
)

baseline_files = [
    "qwen3_zero_shot_validation_results.jsonl",
    "qwen3_few_shot_validation_results.jsonl"
]

for filename in baseline_files:

    path = os.path.join(BASELINE_DIR, filename)

    print("\n" + "=" * 65)
    print("FILE:", filename)

    with open(path, "r", encoding="utf-8") as f:
        first_record = json.loads(f.readline())

    print("Available keys:", list(first_record.keys()))
    print(json.dumps(first_record, indent=2, ensure_ascii=False)[:2500])


In [ ]:

# Cell 32: Compare zero-shot, few-shot and LoRA on failure cases

def load_jsonl(path):
    records = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))

    return records


zero_shot_results = load_jsonl(
    os.path.join(
        BASELINE_DIR,
        "qwen3_zero_shot_validation_results.jsonl"
    )
)

few_shot_results = load_jsonl(
    os.path.join(
        BASELINE_DIR,
        "qwen3_few_shot_validation_results.jsonl"
    )
)


comparison_rows = []

for _, case in false_abstention_examples.iterrows():

    doc_id = case["document_id"]
    engine = case["ocr_engine"]
    field = case["field"]
    expected = str(case["expected"])

    # Find matching predictions from each approach
    zero_record = next(
        r for r in zero_shot_results
        if r["document_id"] == doc_id
        and r["ocr_engine"] == engine
    )

    few_record = next(
        r for r in few_shot_results
        if r["document_id"] == doc_id
        and r["ocr_engine"] == engine
    )

    lora_record = next(
        r for r in lora_validation_results
        if r["document_id"] == doc_id
        and r["ocr_engine"] == engine
    )

    zero_pred = str(zero_record["prediction"].get(field, ""))
    few_pred = str(few_record["prediction"].get(field, ""))
    lora_pred = str(lora_record["prediction"].get(field, ""))

    def is_correct(predicted, expected, field):
        return (
            normalize_field_value(predicted, field)
            ==
            normalize_field_value(expected, field)
        )

    comparison_rows.append({
        "document_id": doc_id,
        "ocr_engine": engine,
        "field": field,
        "expected": expected,

        "zero_shot": zero_pred,
        "zero_shot_correct": is_correct(
            zero_pred, expected, field
        ),

        "few_shot": few_pred,
        "few_shot_correct": is_correct(
            few_pred, expected, field
        ),

        "lora": lora_pred,
        "lora_correct": is_correct(
            lora_pred, expected, field
        )
    })


failure_comparison_df = pd.DataFrame(comparison_rows)

print("COMPARISON OF FALSE ABSTENTION CASES")
display(failure_comparison_df)


# Calculate correct predictions for each approach
print("\nCORRECT EXTRACTIONS ON THESE CASES")

for approach in ["zero_shot", "few_shot", "lora"]:

    correct = failure_comparison_df[
        f"{approach}_correct"
    ].sum()

    total = len(failure_comparison_df)

    print(
        f"{approach}: {correct}/{total} "
        f"({correct / total * 100:.2f}%)"
    )


# Save comparison
failure_comparison_df.to_csv(
    os.path.join(
        METRICS_DIR,
        "three_model_false_abstention_comparison.csv"
    ),
    index=False
)

print("\nComparison saved successfully.")


In [29]:

# Cell 33: Inspect Hindi training examples

hindi_categories = ["invoice", "application", "certificate"]

for category in hindi_categories:

    print("\n" + "=" * 80)
    print("CATEGORY:", category.upper())

    examples = [
        example for example in train_examples
        if example["language"] == "hindi"
        and example["category"] == category
    ]

    # Select one document with all available OCR-engine versions
    selected_doc = examples[0]["document_id"]

    document_examples = [
        example for example in examples
        if example["document_id"] == selected_doc
    ]

    print("DOCUMENT:", selected_doc)

    for example in document_examples:

        print("\nOCR ENGINE:", example["ocr_engine"])

        print("\nOCR INPUT:")
        print(example["input_text"])

        print("\nTRAINING TARGET:")
        print(example["target_json"])

print("\nHindi training inspection completed.")



CATEGORY: INVOICE
DOCUMENT: hindi_invoice_031

OCR ENGINE: tesseract

OCR INPUT:
You are an information extraction system.

Document category: invoice
Document language: hindi

Extract these fields: name, date, document_id, address, amount.
Return exactly these fields as valid JSON.
Extract only information supported by the OCR text.
If a field is missing or unreadable, return an empty string.

OCR TEXT:
चालान

ग्राहक का नाम: मीरा रेड्डी
चालान दिनांक: 34/04/2026
चालान संख्या: ]9४-003॥
पता: बेंगलुरु

कुल राशि: 5747

TRAINING TARGET:
{
  "name": "मीरा रेड्डी",
  "date": "",
  "document_id": "",
  "address": "बेंगलुरु",
  "amount": "5747"
}

OCR ENGINE: easyocr

OCR INPUT:
You are an information extraction system.

Document category: invoice
Document language: hindi

Extract these fields: name, date, document_id, address, amount.
Return exactly these fields as valid JSON.
Extract only information supported by the OCR text.
If a field is missing or unreadable, return an empty string.

OCR

In [30]:

# Cell 34: Compare training targets across OCR engines

target_comparison = []

for document_id in sorted(set(
    example["document_id"] for example in train_examples
)):

    document_examples = [
        example for example in train_examples
        if example["document_id"] == document_id
    ]

    if len(document_examples) != 3:
        continue

    category = document_examples[0]["category"]
    fields = CATEGORY_SCHEMAS[category]

    targets = {}

    for example in document_examples:

        engine = example["ocr_engine"]

        targets[engine] = json.loads(
            example["target_json"]
        )

    engines = sorted(targets.keys())

    for field in fields:

        values = {
            engine: targets[engine].get(field, "")
            for engine in engines
        }

        distinct_values = set(values.values())

        target_comparison.append({
            "document_id": document_id,
            "language": document_examples[0]["language"],
            "category": category,
            "field": field,
            "easyocr": values.get("easyocr", ""),
            "paddleocr": values.get("paddleocr", ""),
            "tesseract": values.get("tesseract", ""),
            "target_consistent": len(distinct_values) == 1
        })

target_comparison_df = pd.DataFrame(target_comparison)

print("TRAINING TARGET CONSISTENCY")
print("=" * 50)

print("Total document-field groups:", len(target_comparison_df))

print(
    "Consistent targets:",
    target_comparison_df["target_consistent"].sum()
)

print(
    "Inconsistent targets:",
    (~target_comparison_df["target_consistent"]).sum()
)

print(
    "Consistency percentage:",
    target_comparison_df["target_consistent"].mean() * 100
)

print("\nLANGUAGE-WISE CONSISTENCY")

language_consistency = (
    target_comparison_df.groupby("language")
    ["target_consistent"]
    .agg(["count", "sum", "mean"])
)

language_consistency["mean"] *= 100

display(language_consistency.round(2))

print("\nEXAMPLES OF INCONSISTENT TARGETS")

display(
    target_comparison_df[
        ~target_comparison_df["target_consistent"]
    ].head(25)
)

target_comparison_df.to_csv(
    os.path.join(
        METRICS_DIR,
        "training_target_consistency_audit.csv"
    ),
    index=False
)

print("\nAudit saved successfully.")


TRAINING TARGET CONSISTENCY
Total document-field groups: 270
Consistent targets: 141
Inconsistent targets: 129
Consistency percentage: 52.22222222222223

LANGUAGE-WISE CONSISTENCY


,count,sum,mean
language,,,
english,90,88,97.78
hindi,90,39,43.33
kannada,90,14,15.56



EXAMPLES OF INCONSISTENT TARGETS


,document_id,language,category,field,easyocr,paddleocr,tesseract,target_consistent
39,english_certificate_022,english,certificate,grade,A+,A+,,False
44,english_certificate_023,english,certificate,grade,B+,B+,,False
91,hindi_application_041,hindi,application,date,10/02/2026,10/02/2026,,False
92,hindi_application_041,hindi,application,document_id,APP-0041,APP-0041,,False
93,hindi_application_041,hindi,application,address,बेंगलुरु,,बेंगलुरु,False
94,hindi_application_041,hindi,application,phone,9000005617,9000005617,,False
95,hindi_application_042,hindi,application,name,,,राहुल रेड्डी,False
96,hindi_application_042,hindi,application,date,11/02/2026,11/02/2026,,False
97,hindi_application_042,hindi,application,document_id,,APP-0042,,False
98,hindi_application_042,hindi,application,address,,मैसूर,मैसूर,False



Audit saved successfully.


In [31]:

# Cell 35: Classify the types of training-target inconsistency

consistency_details = []

for _, row in target_comparison_df.iterrows():

    values = [
        str(row["easyocr"]).strip(),
        str(row["paddleocr"]).strip(),
        str(row["tesseract"]).strip()
    ]

    nonempty_values = [v for v in values if v != ""]

    unique_nonempty = set(nonempty_values)

    if len(nonempty_values) == 0:
        category = "all_empty"

    elif len(nonempty_values) == 3 and len(unique_nonempty) == 1:
        category = "all_same_nonempty"

    elif len(unique_nonempty) == 1:
        category = "availability_inconsistency"

    else:
        category = "nonempty_value_disagreement"

    consistency_details.append({
        **row.to_dict(),
        "consistency_type": category,
        "nonempty_engine_count": len(nonempty_values),
        "unique_nonempty_values": len(unique_nonempty)
    })

consistency_detail_df = pd.DataFrame(consistency_details)

print("TARGET CONSISTENCY TYPE BREAKDOWN")
display(
    consistency_detail_df["consistency_type"]
    .value_counts()
    .rename_axis("consistency_type")
    .to_frame("count")
)

print("\nLANGUAGE-WISE BREAKDOWN")

display(
    pd.crosstab(
        consistency_detail_df["language"],
        consistency_detail_df["consistency_type"]
    )
)

print("\nNON-EMPTY VALUE DISAGREEMENTS")

display(
    consistency_detail_df[
        consistency_detail_df["consistency_type"]
        == "nonempty_value_disagreement"
    ].head(30)
)

consistency_detail_df.to_csv(
    os.path.join(
        METRICS_DIR,
        "training_target_consistency_types.csv"
    ),
    index=False
)

print("\nDetailed audit saved.")


TARGET CONSISTENCY TYPE BREAKDOWN


,count
consistency_type,
all_same_nonempty,134
availability_inconsistency,129
all_empty,7



LANGUAGE-WISE BREAKDOWN


consistency_type,all_empty,all_same_nonempty,availability_inconsistency
language,,,
english,0,88,2
hindi,0,39,51
kannada,7,7,76



NON-EMPTY VALUE DISAGREEMENTS


,document_id,language,category,field,easyocr,paddleocr,tesseract,target_consistent,consistency_type,nonempty_engine_count,unique_nonempty_values



Detailed audit saved.


In [32]:

# Cell 36: Evidence-aware training data audit

from difflib import SequenceMatcher
import pandas as pd
import numpy as np
import re
import os

def normalize_for_similarity(value, field):

    value = unicodedata.normalize("NFC", str(value))
    value = value.replace("\u200c", "").replace("\u200d", "")
    value = value.strip().casefold()

    if field in ["date", "phone", "amount", "document_id"]:
        value = re.sub(r"[\s/\\\-:.,|]", "", value)
    else:
        value = re.sub(r"\s+", " ", value)

    return value


def best_substring_similarity(expected, ocr_text, field):

    expected_norm = normalize_for_similarity(expected, field)
    ocr_norm = normalize_for_similarity(ocr_text, field)

    if not expected_norm or not ocr_norm:
        return 0.0

    if expected_norm in ocr_norm:
        return 1.0

    target_length = len(expected_norm)

    # Search OCR windows around the expected length.
    min_length = max(1, int(target_length * 0.70))
    max_length = min(
        len(ocr_norm),
        max(target_length + 5, int(target_length * 1.30))
    )

    best_score = 0.0

    for window_length in range(min_length, max_length + 1):

        for start in range(len(ocr_norm) - window_length + 1):

            candidate = ocr_norm[start:start + window_length]

            score = SequenceMatcher(
                None,
                expected_norm,
                candidate
            ).ratio()

            best_score = max(best_score, score)

    return best_score


evidence_rows = []

for record in train_records:

    fields = CATEGORY_SCHEMAS[record["category"]]

    for field in fields:

        expected = str(record["ground_truth"].get(field, ""))
        ocr_text = record["ocr_text"]

        exact_match = is_field_supported(
            expected,
            ocr_text,
            field
        )

        similarity = best_substring_similarity(
            expected,
            ocr_text,
            field
        )

        if exact_match:
            evidence_category = "exact_match"

        elif similarity >= 0.75:
            evidence_category = "possible_recoverable"

        else:
            evidence_category = "unavailable_or_uncertain"

        evidence_rows.append({
            "document_id": record["document_id"],
            "language": record["language"],
            "category": record["category"],
            "ocr_engine": record["ocr_engine"],
            "field": field,
            "expected": expected,
            "ocr_text": ocr_text,
            "exact_match": exact_match,
            "similarity_score": similarity,
            "evidence_category": evidence_category
        })


evidence_audit_df = pd.DataFrame(evidence_rows)

print("EVIDENCE AUDIT SUMMARY")
print("=" * 50)

print("Total field records:", len(evidence_audit_df))

display(
    evidence_audit_df["evidence_category"]
    .value_counts()
    .to_frame("count")
)

print("\nLANGUAGE-WISE EVIDENCE DISTRIBUTION")

display(
    pd.crosstab(
        evidence_audit_df["language"],
        evidence_audit_df["evidence_category"]
    )
)

print("\nPOSSIBLE RECOVERABLE CASES")

display(
    evidence_audit_df[
        evidence_audit_df["evidence_category"]
        == "possible_recoverable"
    ][
        [
            "document_id",
            "language",
            "ocr_engine",
            "field",
            "expected",
            "similarity_score"
        ]
    ].head(30)
)

AUDIT_PATH = os.path.join(
    METRICS_DIR,
    "experiment2_evidence_audit.csv"
)

evidence_audit_df.to_csv(AUDIT_PATH, index=False)

print("\nAudit saved to:", AUDIT_PATH)


EVIDENCE AUDIT SUMMARY
Total field records: 810


,count
evidence_category,
exact_match,621
unavailable_or_uncertain,109
possible_recoverable,80



LANGUAGE-WISE EVIDENCE DISTRIBUTION


evidence_category,exact_match,possible_recoverable,unavailable_or_uncertain
language,,,
english,268,0,2
hindi,205,33,32
kannada,148,47,75



POSSIBLE RECOVERABLE CASES


,document_id,language,ocr_engine,field,expected,similarity_score
91,hindi_invoice_031,hindi,tesseract,date,31/01/2026,0.750000
96,hindi_invoice_032,hindi,tesseract,date,01/02/2026,0.875000
104,hindi_invoice_033,hindi,tesseract,amount,6021,0.857143
109,hindi_invoice_034,hindi,tesseract,amount,6158,0.857143
121,hindi_application_041,hindi,tesseract,date,10/02/2026,0.933333
124,hindi_application_041,hindi,tesseract,phone,9000005617,0.900000
126,hindi_application_042,hindi,tesseract,date,11/02/2026,0.857143
131,hindi_application_043,hindi,tesseract,date,12/02/2026,0.933333
134,hindi_application_043,hindi,tesseract,phone,9000005891,0.947368
136,hindi_application_044,hindi,tesseract,date,13/02/2026,0.933333



Audit saved to: /content/drive/MyDrive/BharatDocAI/outputs/metrics/finetuning/experiment2_evidence_audit.csv


In [33]:

# Cell 37: Generate manual verification sheet

from difflib import SequenceMatcher
import pandas as pd
import os

def find_best_ocr_candidate(expected, ocr_text, field):

    expected_norm = normalize_for_similarity(expected, field)
    ocr_norm = normalize_for_similarity(ocr_text, field)

    if not expected_norm or not ocr_norm:
        return "", 0.0

    if expected_norm in ocr_norm:
        return expected_norm, 1.0

    target_length = len(expected_norm)

    min_length = max(1, int(target_length * 0.70))
    max_length = min(
        len(ocr_norm),
        max(target_length + 5, int(target_length * 1.30))
    )

    best_candidate = ""
    best_score = 0.0

    for window_length in range(min_length, max_length + 1):

        for start in range(len(ocr_norm) - window_length + 1):

            candidate = ocr_norm[start:start + window_length]

            score = SequenceMatcher(
                None,
                expected_norm,
                candidate
            ).ratio()

            if score > best_score:
                best_score = score
                best_candidate = candidate

    return best_candidate, best_score


# Select only possible recoverable cases
review_df = evidence_audit_df[
    evidence_audit_df["evidence_category"] == "possible_recoverable"
].copy()

candidate_results = []

for _, row in review_df.iterrows():

    candidate, score = find_best_ocr_candidate(
        row["expected"],
        row["ocr_text"],
        row["field"]
    )

    candidate_results.append({
        "document_id": row["document_id"],
        "language": row["language"],
        "ocr_engine": row["ocr_engine"],
        "field": row["field"],
        "expected_value": row["expected"],
        "ocr_candidate": candidate,
        "similarity_score": round(score, 4),
        "ocr_text": row["ocr_text"],
        "manual_decision": "",
        "verified_target": "",
        "reviewer_notes": ""
    })


manual_review_df = pd.DataFrame(candidate_results)

# Sort by language, then highest similarity
manual_review_df = manual_review_df.sort_values(
    ["language", "similarity_score"],
    ascending=[True, False]
).reset_index(drop=True)


REVIEW_PATH = os.path.join(
    METRICS_DIR,
    "experiment2_manual_evidence_review.csv"
)

manual_review_df.to_csv(REVIEW_PATH, index=False)

print("Manual review records:", len(manual_review_df))

print("\nCANDIDATE DISTRIBUTION")
display(
    manual_review_df.groupby("language")
    .size()
    .to_frame("candidates")
)

print("\nFIRST 30 CANDIDATES")
display(
    manual_review_df[
        [
            "document_id",
            "language",
            "ocr_engine",
            "field",
            "expected_value",
            "ocr_candidate",
            "similarity_score"
        ]
    ].head(30)
)

print("\nReview file saved to:")
print(REVIEW_PATH)


Manual review records: 80

CANDIDATE DISTRIBUTION


,candidates
language,
hindi,33
kannada,47



FIRST 30 CANDIDATES


,document_id,language,ocr_engine,field,expected_value,ocr_candidate,similarity_score
0,hindi_certificate_051,hindi,paddleocr,institution,राष्ट्रीय प्रौद्योगिकी संस्थान,राष्रीय प्रौद्योगिकी संस्थान,0.9655
1,hindi_certificate_055,hindi,paddleocr,institution,राष्ट्रीय प्रौद्योगिकी संस्थान,राष्रीय प्रौद्योगिकी संस्थान,0.9655
2,hindi_invoice_032,hindi,paddleocr,name,अर्जुन वर्मा,अर्जन वर्मा,0.9565
3,hindi_application_043,hindi,tesseract,phone,9000005891,900000589,0.9474
4,hindi_application_041,hindi,tesseract,date,10/02/2026,0022026,0.9333
5,hindi_application_043,hindi,tesseract,date,12/02/2026,2022026,0.9333
6,hindi_application_044,hindi,tesseract,date,13/02/2026,3022026,0.9333
7,hindi_application_045,hindi,tesseract,date,14/02/2026,4022026,0.9333
8,hindi_application_046,hindi,tesseract,date,15/02/2026,5022026,0.9333
9,hindi_certificate_052,hindi,tesseract,date,21/02/2026,2022026,0.9333



Review file saved to:
/content/drive/MyDrive/BharatDocAI/outputs/metrics/finetuning/experiment2_manual_evidence_review.csv


In [34]:

# Cell 38: Automated evidence verification using OCR agreement

from difflib import SequenceMatcher
import pandas as pd
import json
import os

# Index all training OCR records
records_by_document = {}

for record in train_records:
    records_by_document.setdefault(
        record["document_id"], {}
    )[record["ocr_engine"]] = record


automated_results = []

for _, row in evidence_audit_df.iterrows():

    doc_id = row["document_id"]
    field = row["field"]
    current_engine = row["ocr_engine"]
    expected = str(row["expected"])

    candidate, similarity = find_best_ocr_candidate(
        expected,
        row["ocr_text"],
        field
    )

    # Check whether another OCR engine exactly supports this field
    other_engine_support = []

    document_records = records_by_document.get(doc_id, {})

    for engine, other_record in document_records.items():

        if engine == current_engine:
            continue

        other_text = other_record["ocr_text"]

        if is_field_supported(expected, other_text, field):
            other_engine_support.append(engine)

    cross_ocr_confirmed = len(other_engine_support) > 0

    if row["exact_match"]:

        decision = "exact_supported"
        target = expected

    elif cross_ocr_confirmed and similarity >= 0.90:

        decision = "high_confidence_candidate"
        target = expected

    elif cross_ocr_confirmed and similarity >= 0.75:

        decision = "medium_confidence_candidate"
        target = ""

    elif similarity >= 0.75:

        decision = "low_confidence_candidate"
        target = ""

    else:

        decision = "unavailable_or_uncertain"
        target = ""

    automated_results.append({
        "document_id": doc_id,
        "language": row["language"],
        "category": row["category"],
        "ocr_engine": current_engine,
        "field": field,
        "expected": expected,
        "ocr_candidate": candidate,
        "similarity": round(similarity, 4),
        "other_engines_supporting": ", ".join(other_engine_support),
        "cross_ocr_confirmed": cross_ocr_confirmed,
        "automated_decision": decision,
        "proposed_target": target
    })


automated_review_df = pd.DataFrame(automated_results)

print("AUTOMATED EVIDENCE CLASSIFICATION")
print("=" * 55)

display(
    automated_review_df["automated_decision"]
    .value_counts()
    .to_frame("count")
)

print("\nLANGUAGE-WISE CLASSIFICATION")

display(
    pd.crosstab(
        automated_review_df["language"],
        automated_review_df["automated_decision"]
    )
)

print("\nHIGH-CONFIDENCE CANDIDATES")

display(
    automated_review_df[
        automated_review_df["automated_decision"]
        == "high_confidence_candidate"
    ][
        [
            "document_id",
            "language",
            "ocr_engine",
            "field",
            "expected",
            "ocr_candidate",
            "similarity",
            "other_engines_supporting"
        ]
    ].head(30)
)


# Save separately. Do not overwrite Experiment 1 data.
AUTO_PATH = os.path.join(
    METRICS_DIR,
    "experiment2_automated_evidence_classification.csv"
)

automated_review_df.to_csv(AUTO_PATH, index=False)

print("\nSaved:", AUTO_PATH)


AUTOMATED EVIDENCE CLASSIFICATION


,count
automated_decision,
exact_supported,621
unavailable_or_uncertain,109
medium_confidence_candidate,53
high_confidence_candidate,19
low_confidence_candidate,8



LANGUAGE-WISE CLASSIFICATION


automated_decision,exact_supported,high_confidence_candidate,low_confidence_candidate,medium_confidence_candidate,unavailable_or_uncertain
language,,,,,
english,268,0,0,0,2
hindi,205,17,0,16,32
kannada,148,2,8,37,75



HIGH-CONFIDENCE CANDIDATES


,document_id,language,ocr_engine,field,expected,ocr_candidate,similarity,other_engines_supporting
121,hindi_application_041,hindi,tesseract,date,10/02/2026,0022026,0.9333,"easyocr, paddleocr"
124,hindi_application_041,hindi,tesseract,phone,9000005617,9000005647,0.9000,"easyocr, paddleocr"
131,hindi_application_043,hindi,tesseract,date,12/02/2026,2022026,0.9333,"easyocr, paddleocr"
134,hindi_application_043,hindi,tesseract,phone,9000005891,900000589,0.9474,"easyocr, paddleocr"
136,hindi_application_044,hindi,tesseract,date,13/02/2026,3022026,0.9333,"easyocr, paddleocr"
141,hindi_application_045,hindi,tesseract,date,14/02/2026,4022026,0.9333,"easyocr, paddleocr"
144,hindi_application_045,hindi,tesseract,phone,9000006165,9000006465,0.9000,"easyocr, paddleocr"
146,hindi_application_046,hindi,tesseract,date,15/02/2026,5022026,0.9333,"easyocr, paddleocr"
156,hindi_certificate_052,hindi,tesseract,date,21/02/2026,2022026,0.9333,"easyocr, paddleocr"
367,hindi_application_042,hindi,easyocr,document_id,APP-0042,pp0042,0.9231,paddleocr



Saved: /content/drive/MyDrive/BharatDocAI/outputs/metrics/finetuning/experiment2_automated_evidence_classification.csv


In [35]:

# Cell 39: Field-context verification for high-confidence candidates

import pandas as pd
import re
import unicodedata
from difflib import SequenceMatcher

# Labels commonly used in our synthetic documents.
# These are candidate labels, not exhaustive language dictionaries.

FIELD_LABELS = {
    "name": [
        "name", "customer name", "applicant name", "recipient name",
        "ग्राहक का नाम", "आवेदक का नाम", "प्राप्तकर्ता का नाम",
        "ಹೆಸರು", "ಅರ್ಜಿದಾರರ ಹೆಸರು", "ಗ್ರಾಹಕರ ಹೆಸರು"
    ],
    "date": [
        "date", "invoice date", "application date", "issue date",
        "चालान दिनांक", "आवेदन दिनांक", "जारी करने की तारीख",
        "ದಿನಾಂಕ", "ಅರ್ಜಿ ದಿನಾಂಕ", "ಜಾರಿ ದಿನಾಂಕ"
    ],
    "document_id": [
        "document number", "invoice number", "application number",
        "certificate number", "चालान संख्या", "आवेदन संख्या",
        "प्रमाण पत्र संख्या", "ದಾಖಲೆ ಸಂಖ್ಯೆ", "ಅರ್ಜಿ ಸಂಖ್ಯೆ",
        "ಪ್ರಮಾಣ ಪತ್ರ ಸಂಖ್ಯೆ"
    ],
    "address": [
        "address", "पता", "ವಿಳಾಸ"
    ],
    "amount": [
        "amount", "total amount", "कुल राशि", "ರಾಶಿ", "ಒಟ್ಟು ಮೊತ್ತ"
    ],
    "phone": [
        "phone number", "phone", "फोन नंबर", "मोबाइल नंबर",
        "ದೂರವಾಣಿ ಸಂಖ್ಯೆ", "ಮೊಬೈಲ್ ಸಂಖ್ಯೆ"
    ],
    "grade": [
        "grade", "श्रेणी", "ಅಂಕ", "ಶ್ರೇಣಿ"
    ],
    "institution": [
        "institution", "संस्थान", "ಸಂಸ್ಥೆ"
    ]
}


def normalize_line(text):
    text = unicodedata.normalize("NFC", str(text))
    text = text.replace("\u200c", "").replace("\u200d", "")
    return re.sub(r"\s+", " ", text.strip().casefold())


def find_label_line(lines, field):

    labels = FIELD_LABELS.get(field, [])

    for index, line in enumerate(lines):

        normalized_line = normalize_line(line)

        for label in labels:

            normalized_label = normalize_line(label)

            if normalized_label in normalized_line:
                return index, label

    return None, None


def verify_field_context(ocr_text, field, expected):

    lines = [
        line.strip()
        for line in str(ocr_text).splitlines()
        if line.strip()
    ]

    label_index, matched_label = find_label_line(lines, field)

    if label_index is None:
        return {
            "label_found": False,
            "matched_label": "",
            "context_similarity": 0.0,
            "context_evidence": "",
            "context_verified": False
        }

    # Inspect the label line and up to three following lines.
    nearby_lines = lines[label_index:label_index + 4]

    best_score = 0.0
    best_line = ""

    for line in nearby_lines:

        score = SequenceMatcher(
            None,
            normalize_for_similarity(expected, field),
            normalize_for_similarity(line, field)
        ).ratio()

        if score > best_score:
            best_score = score
            best_line = line

    # Context presence is supporting evidence only.
    # It does not automatically establish correctness.
    context_verified = best_score >= 0.70

    return {
        "label_found": True,
        "matched_label": matched_label,
        "context_similarity": round(best_score, 4),
        "context_evidence": best_line,
        "context_verified": context_verified
    }


# Read the 19 high-confidence candidates.
high_candidates = automated_review_df[
    automated_review_df["automated_decision"]
    == "high_confidence_candidate"
].copy()

context_results = []

for _, row in high_candidates.iterrows():

    doc_id = row["document_id"]
    engine = row["ocr_engine"]
    field = row["field"]
    expected = row["expected"]

    source_record = records_by_document[doc_id][engine]

    context = verify_field_context(
        source_record["ocr_text"],
        field,
        expected
    )

    context_results.append({
        **row.to_dict(),
        **context
    })


context_review_df = pd.DataFrame(context_results)

# Conservative classification
def classify_context(row):

    if not row["label_found"]:
        return "needs_review"

    if row["context_verified"]:
        return "context_supported_candidate"

    return "needs_review"


context_review_df["context_decision"] = (
    context_review_df.apply(classify_context, axis=1)
)

print("FIELD-CONTEXT VERIFICATION")
print("=" * 55)

display(
    context_review_df["context_decision"]
    .value_counts()
    .to_frame("count")
)

print("\nDETAILED RESULTS")

display(
    context_review_df[
        [
            "document_id",
            "language",
            "ocr_engine",
            "field",
            "expected",
            "ocr_candidate",
            "other_engines_supporting",
            "matched_label",
            "context_evidence",
            "context_similarity",
            "context_decision"
        ]
    ]
)

# Save without overwriting previous audit files.
CONTEXT_PATH = os.path.join(
    METRICS_DIR,
    "experiment2_context_verification.csv"
)

context_review_df.to_csv(CONTEXT_PATH, index=False)

print("\nContext verification saved:", CONTEXT_PATH)


FIELD-CONTEXT VERIFICATION


,count
context_decision,
needs_review,10
context_supported_candidate,9



DETAILED RESULTS


,document_id,language,ocr_engine,field,expected,ocr_candidate,other_engines_supporting,matched_label,context_evidence,context_similarity,context_decision
0,hindi_application_041,hindi,tesseract,date,10/02/2026,0022026,"easyocr, paddleocr",आवेदन दिनांक,,0.0000,needs_review
1,hindi_application_041,hindi,tesseract,phone,9000005617,9000005647,"easyocr, paddleocr",फोन नंबर,0/02/2026,0.4706,needs_review
2,hindi_application_043,hindi,tesseract,date,12/02/2026,2022026,"easyocr, paddleocr",आवेदन दिनांक,आवेदन दिनांक 2/02/2026,0.5385,needs_review
3,hindi_application_043,hindi,tesseract,phone,9000005891,900000589,"easyocr, paddleocr",फोन नंबर,फोन नंबर 900000589|,0.6923,needs_review
4,hindi_application_044,hindi,tesseract,date,13/02/2026,3022026,"easyocr, paddleocr",आवेदन दिनांक,आवेदन दिनांक 3/02/2026,0.5385,needs_review
5,hindi_application_045,hindi,tesseract,date,14/02/2026,4022026,"easyocr, paddleocr",आवेदन दिनांक,4/02/2026,0.9333,context_supported_candidate
6,hindi_application_045,hindi,tesseract,phone,9000006165,9000006465,"easyocr, paddleocr",फोन नंबर,9000006465,0.9000,context_supported_candidate
7,hindi_application_046,hindi,tesseract,date,15/02/2026,5022026,"easyocr, paddleocr",आवेदन दिनांक,2. आवेदन दिनांक: 5/02/2026,0.5185,needs_review
8,hindi_certificate_052,hindi,tesseract,date,21/02/2026,2022026,"easyocr, paddleocr",जारी करने की तारीख,,0.0000,needs_review
9,hindi_application_042,hindi,easyocr,document_id,APP-0042,pp0042,paddleocr,आवेदन संख्या,^PP-0042,0.8571,context_supported_candidate



Context verification saved: /content/drive/MyDrive/BharatDocAI/outputs/metrics/finetuning/experiment2_context_verification.csv


In [36]:

# Cell 40: Stricter field-value association

def extract_value_near_label(ocr_text, field):

    lines = [
        line.strip()
        for line in str(ocr_text).splitlines()
        if line.strip()
    ]

    label_index, matched_label = find_label_line(lines, field)

    if label_index is None:
        return "", ""

    labels = FIELD_LABELS.get(field, [])

    # Remove the recognized label from its line.
    current_line = lines[label_index]

    for label in sorted(labels, key=len, reverse=True):

        pattern = re.compile(re.escape(label), re.IGNORECASE)

        current_line = pattern.sub("", current_line, count=1)

    current_line = re.sub(r"^[\s:：|.।]+", "", current_line).strip()

    if current_line:
        return current_line, matched_label

    # If the label is on a separate line, inspect the next line.
    if label_index + 1 < len(lines):

        next_line = lines[label_index + 1]

        next_is_label, _ = find_label_line([next_line], field)

        if next_is_label is None:
            return next_line, matched_label

    return "", matched_label


strict_results = []

for _, row in high_candidates.iterrows():

    doc_id = row["document_id"]
    engine = row["ocr_engine"]
    field = row["field"]
    expected = str(row["expected"])

    source_record = records_by_document[doc_id][engine]

    extracted_value, matched_label = extract_value_near_label(
        source_record["ocr_text"],
        field
    )

    extracted_norm = normalize_for_similarity(
        extracted_value, field
    )

    expected_norm = normalize_for_similarity(
        expected, field
    )

    exact_context_match = (
        extracted_norm == expected_norm
        and extracted_norm != ""
    )

    # Check exact expected-value support in other OCR engines.
    supporting_engines = []

    for other_engine, other_record in records_by_document[doc_id].items():

        if other_engine == engine:
            continue

        if is_field_supported(
            expected,
            other_record["ocr_text"],
            field
        ):
            supporting_engines.append(other_engine)

    independent_support = len(supporting_engines) > 0

    # Strict eligibility, not automatic correction.
    eligible = (
        exact_context_match
        and independent_support
    )

    strict_results.append({
        "document_id": doc_id,
        "language": row["language"],
        "ocr_engine": engine,
        "field": field,
        "expected": expected,
        "extracted_near_label": extracted_value,
        "label": matched_label,
        "exact_context_match": exact_context_match,
        "independent_ocr_support": independent_support,
        "supporting_engines": ", ".join(supporting_engines),
        "eligible_for_further_review": eligible
    })


strict_review_df = pd.DataFrame(strict_results)

print("STRICT VERIFICATION RESULTS")

display(
    strict_review_df[
        [
            "document_id",
            "field",
            "expected",
            "extracted_near_label",
            "exact_context_match",
            "independent_ocr_support",
            "eligible_for_further_review"
        ]
    ]
)

print("\nEligibility summary:")
display(
    strict_review_df["eligible_for_further_review"]
    .value_counts()
    .to_frame("count")
)

STRICT_PATH = os.path.join(
    METRICS_DIR,
    "experiment2_strict_field_verification.csv"
)

strict_review_df.to_csv(STRICT_PATH, index=False)

print("\nSaved:", STRICT_PATH)


STRICT VERIFICATION RESULTS


,document_id,field,expected,extracted_near_label,exact_context_match,independent_ocr_support,eligible_for_further_review
0,hindi_application_041,date,10/02/2026,आवेदन संख्या:,False,True,False
1,hindi_application_041,phone,9000005617,ः,False,True,False
2,hindi_application_043,date,12/02/2026,2/02/2026,False,True,False
3,hindi_application_043,phone,9000005891,900000589|,False,True,False
4,hindi_application_044,date,13/02/2026,3/02/2026,False,True,False
5,hindi_application_045,date,14/02/2026,4/02/2026,False,True,False
6,hindi_application_045,phone,9000006165,9000006465,False,True,False
7,hindi_application_046,date,15/02/2026,2. : 5/02/2026,False,True,False
8,hindi_certificate_052,date,21/02/2026,प्रमाण पत्र संख्या:,False,True,False
9,hindi_application_042,document_id,APP-0042,ः,False,True,False



Eligibility summary:


,count
eligible_for_further_review,
False,19



Saved: /content/drive/MyDrive/BharatDocAI/outputs/metrics/finetuning/experiment2_strict_field_verification.csv


In [37]:

import re
import unicodedata
from difflib import SequenceMatcher
from collections import defaultdict
import pandas as pd

def normalize_for_matching(value):
    value = unicodedata.normalize("NFC", str(value))
    value = value.replace("\u200c", "").replace("\u200d", "")
    value = value.casefold().strip()
    value = re.sub(r"\s+", "", value)
    return value

def best_fuzzy_window(expected, ocr_text):
    expected_norm = normalize_for_matching(expected)
    text_norm = normalize_for_matching(ocr_text)

    if not expected_norm or not text_norm:
        return 0.0, ""

    if expected_norm in text_norm:
        return 1.0, expected_norm

    target_len = len(expected_norm)
    best_score = 0.0
    best_candidate = ""

    min_len = max(1, target_len - max(2, int(target_len * 0.20)))
    max_len = min(
        len(text_norm),
        target_len + max(2, int(target_len * 0.20))
    )

    for window_len in range(min_len, max_len + 1):
        for start in range(len(text_norm) - window_len + 1):
            candidate = text_norm[start:start + window_len]
            score = SequenceMatcher(
                None, expected_norm, candidate
            ).ratio()

            if score > best_score:
                best_score = score
                best_candidate = candidate

    return best_score, best_candidate


# Group training records by original document
records_by_document = defaultdict(list)

for record in train_records:
    records_by_document[record["document_id"]].append(record)


verification_rows = []

for document_id, doc_records in records_by_document.items():

    # Build lookup for each OCR engine
    engine_records = {
        record["ocr_engine"]: record
        for record in doc_records
    }

    for record in doc_records:
        engine = record["ocr_engine"]
        ocr_text = record["ocr_text"]
        ground_truth = record["ground_truth"]

        fields = CATEGORY_SCHEMAS[record["category"]]

        for field in fields:
            expected = str(ground_truth.get(field, "")).strip()

            if not expected:
                continue

            current_exact = (
                normalize_for_matching(expected)
                in normalize_for_matching(ocr_text)
            )

            supporting_engines = []

            for other_engine, other_record in engine_records.items():
                if other_engine == engine:
                    continue

                other_text = other_record["ocr_text"]

                if (
                    normalize_for_matching(expected)
                    in normalize_for_matching(other_text)
                ):
                    supporting_engines.append(other_engine)

            similarity, candidate = best_fuzzy_window(
                expected, ocr_text
            )

            if current_exact:
                status = "exact_supported"
            elif supporting_engines and similarity >= 0.80:
                status = "cross_ocr_fuzzy_candidate"
            elif similarity >= 0.90:
                status = "fuzzy_only_candidate"
            else:
                status = "unavailable_or_uncertain"

            verification_rows.append({
                "document_id": document_id,
                "language": record["language"],
                "category": record["category"],
                "ocr_engine": engine,
                "field": field,
                "expected": expected,
                "current_exact_match": current_exact,
                "best_similarity": round(similarity, 4),
                "best_ocr_candidate": candidate,
                "supporting_engines": ", ".join(supporting_engines),
                "status": status
            })


experiment2_verification = pd.DataFrame(verification_rows)

print("Total field records:", len(experiment2_verification))
print("\nEvidence classification:")
display(
    experiment2_verification["status"]
    .value_counts()
    .to_frame("count")
)

print("\nClassification by language:")
display(
    pd.crosstab(
        experiment2_verification["language"],
        experiment2_verification["status"]
    )
)

verification_path = os.path.join(
    METRICS_DIR,
    "experiment2_cross_ocr_fuzzy_verification.csv"
)

experiment2_verification.to_csv(verification_path, index=False)

print("\nSaved:", verification_path)


Total field records: 810

Evidence classification:


,count
status,
exact_supported,598
unavailable_or_uncertain,130
cross_ocr_fuzzy_candidate,74
fuzzy_only_candidate,8



Classification by language:


status,cross_ocr_fuzzy_candidate,exact_supported,fuzzy_only_candidate,unavailable_or_uncertain
language,,,,
english,0,268,0,2
hindi,30,200,0,40
kannada,44,130,8,88



Saved: /content/drive/MyDrive/BharatDocAI/outputs/metrics/finetuning/experiment2_cross_ocr_fuzzy_verification.csv


In [38]:

def normalize_evidence(value, field):
    value = unicodedata.normalize("NFC", str(value))
    value = value.replace("\u200c", "").replace("\u200d", "")
    value = value.strip().casefold()

    if field in ["date", "phone", "amount", "document_id"]:
        value = re.sub(r"[\s/\\\-:.,|]", "", value)
    else:
        value = re.sub(r"\s+", " ", value)

    return value


def field_similarity(expected, candidate, field):
    expected_norm = normalize_evidence(expected, field)
    candidate_norm = normalize_evidence(candidate, field)

    if not expected_norm or not candidate_norm:
        return 0.0

    return SequenceMatcher(
        None, expected_norm, candidate_norm
    ).ratio()


# Recalculate exact support using the consistent normalizer
exact_results = []

for _, row in experiment2_verification.iterrows():
    expected = str(row["expected"])
    field = row["field"]

    doc_records = [
        r for r in train_records
        if r["document_id"] == row["document_id"]
        and r["ocr_engine"] == row["ocr_engine"]
    ]

    ocr_text = doc_records[0]["ocr_text"]

    exact = (
        normalize_evidence(expected, field)
        in normalize_evidence(ocr_text, field)
    )

    exact_results.append(exact)

experiment2_verification["consistent_exact_match"] = exact_results

print("Consistent exact-match count:")
print(experiment2_verification["consistent_exact_match"].sum())

print("\nPrevious versus corrected exact classification:")
print(
    pd.crosstab(
        experiment2_verification["current_exact_match"],
        experiment2_verification["consistent_exact_match"]
    )
)

print("\nLanguage-wise exact support:")
display(
    experiment2_verification.groupby("language")[
        "consistent_exact_match"
    ].agg(["sum", "count", "mean"])
)


Consistent exact-match count:
621

Previous versus corrected exact classification:
consistent_exact_match  False  True 
current_exact_match                 
False                     189     23
True                        0    598

Language-wise exact support:


,sum,count,mean
language,,,
english,268,270,0.992593
hindi,205,270,0.759259
kannada,148,270,0.548148


In [39]:

from collections import defaultdict
from difflib import SequenceMatcher
import pandas as pd

def best_field_match(expected, ocr_text, field):
    expected_norm = normalize_evidence(expected, field)
    text_norm = normalize_evidence(ocr_text, field)

    if not expected_norm or not text_norm:
        return 0.0, ""

    if expected_norm in text_norm:
        return 1.0, expected_norm

    target_len = len(expected_norm)
    tolerance = max(1, int(target_len * 0.20))

    min_len = max(1, target_len - tolerance)
    max_len = min(len(text_norm), target_len + tolerance)

    best_score = 0.0
    best_candidate = ""

    for window_len in range(min_len, max_len + 1):
        for start in range(len(text_norm) - window_len + 1):
            candidate = text_norm[start:start + window_len]

            score = SequenceMatcher(
                None, expected_norm, candidate
            ).ratio()

            if score > best_score:
                best_score = score
                best_candidate = candidate

    return best_score, best_candidate


records_by_document = defaultdict(list)

for record in train_records:
    records_by_document[record["document_id"]].append(record)


cell43_rows = []

for document_id, doc_records in records_by_document.items():

    engine_lookup = {
        record["ocr_engine"]: record
        for record in doc_records
    }

    for record in doc_records:
        engine = record["ocr_engine"]
        fields = CATEGORY_SCHEMAS[record["category"]]

        for field in fields:
            expected = str(
                record["ground_truth"].get(field, "")
            ).strip()

            if not expected:
                continue

            current_text = record["ocr_text"]

            current_norm = normalize_evidence(
                current_text, field
            )
            expected_norm = normalize_evidence(
                expected, field
            )

            exact_match = expected_norm in current_norm

            supporting_engines = []

            for other_engine, other_record in engine_lookup.items():
                if other_engine == engine:
                    continue

                other_norm = normalize_evidence(
                    other_record["ocr_text"], field
                )

                if expected_norm in other_norm:
                    supporting_engines.append(other_engine)

            similarity, candidate = best_field_match(
                expected, current_text, field
            )

            if exact_match:
                status = "exact_supported"
            elif supporting_engines and similarity >= 0.80:
                status = "cross_ocr_fuzzy_candidate"
            elif similarity >= 0.90:
                status = "fuzzy_only_candidate"
            else:
                status = "unavailable_or_uncertain"

            cell43_rows.append({
                "document_id": document_id,
                "language": record["language"],
                "category": record["category"],
                "ocr_engine": engine,
                "field": field,
                "expected": expected,
                "exact_match": exact_match,
                "similarity": round(similarity, 4),
                "candidate_text": candidate,
                "supporting_engines": ", ".join(
                    supporting_engines
                ),
                "status": status
            })


experiment2_cell43 = pd.DataFrame(cell43_rows)

print("Total fields:", len(experiment2_cell43))

print("\nUpdated evidence classification:")
display(
    experiment2_cell43["status"]
    .value_counts()
    .to_frame("count")
)

print("\nLanguage-wise classification:")
display(
    pd.crosstab(
        experiment2_cell43["language"],
        experiment2_cell43["status"]
    )
)

print("\nField-wise classification:")
display(
    pd.crosstab(
        experiment2_cell43["field"],
        experiment2_cell43["status"]
    )
)

# Show candidate records for inspection
candidates = experiment2_cell43[
    experiment2_cell43["status"].isin([
        "cross_ocr_fuzzy_candidate",
        "fuzzy_only_candidate"
    ])
].sort_values(
    ["language", "field", "similarity"],
    ascending=[True, True, False]
)

display(candidates[[
    "document_id",
    "language",
    "ocr_engine",
    "field",
    "expected",
    "candidate_text",
    "similarity",
    "supporting_engines",
    "status"
]].head(40))

# Save the updated audit
cell43_path = os.path.join(
    METRICS_DIR,
    "experiment2_consistent_fuzzy_audit.csv"
)

experiment2_cell43.to_csv(cell43_path, index=False)

print("\nSaved:", cell43_path)


Total fields: 810

Updated evidence classification:


,count
status,
exact_supported,621
unavailable_or_uncertain,130
cross_ocr_fuzzy_candidate,51
fuzzy_only_candidate,8



Language-wise classification:


status,cross_ocr_fuzzy_candidate,exact_supported,fuzzy_only_candidate,unavailable_or_uncertain
language,,,,
english,0,268,0,2
hindi,24,205,0,41
kannada,27,148,8,87



Field-wise classification:


status,cross_ocr_fuzzy_candidate,exact_supported,fuzzy_only_candidate,unavailable_or_uncertain
field,,,,
address,5,93,0,10
amount,3,49,0,2
date,25,135,0,2
document_id,2,94,6,60
grade,0,31,0,23
institution,3,43,2,6
name,9,131,0,22
phone,4,45,0,5


,document_id,language,ocr_engine,field,expected,candidate_text,similarity,supporting_engines,status
373,hindi_application_041,hindi,paddleocr,address,बेंगलुरु,बेंगलुर,0.9333,"tesseract, easyocr",cross_ocr_fuzzy_candidate
358,hindi_invoice_036,hindi,paddleocr,address,शिवमोग्गा,शिवमोगा,0.8235,"tesseract, easyocr",cross_ocr_fuzzy_candidate
448,hindi_application_046,hindi,paddleocr,address,शिवमोग्गा,शिवमोगा,0.8235,"tesseract, easyocr",cross_ocr_fuzzy_candidate
304,hindi_invoice_033,hindi,tesseract,amount,6021,602,0.8571,"easyocr, paddleocr",cross_ocr_fuzzy_candidate
319,hindi_invoice_034,hindi,tesseract,amount,6158,658,0.8571,"easyocr, paddleocr",cross_ocr_fuzzy_candidate
361,hindi_application_041,hindi,tesseract,date,10/02/2026,0022026,0.9333,"easyocr, paddleocr",cross_ocr_fuzzy_candidate
391,hindi_application_043,hindi,tesseract,date,12/02/2026,2022026,0.9333,"easyocr, paddleocr",cross_ocr_fuzzy_candidate
406,hindi_application_044,hindi,tesseract,date,13/02/2026,3022026,0.9333,"easyocr, paddleocr",cross_ocr_fuzzy_candidate
421,hindi_application_045,hindi,tesseract,date,14/02/2026,4022026,0.9333,"easyocr, paddleocr",cross_ocr_fuzzy_candidate
436,hindi_application_046,hindi,tesseract,date,15/02/2026,5022026,0.9333,"easyocr, paddleocr",cross_ocr_fuzzy_candidate



Saved: /content/drive/MyDrive/BharatDocAI/outputs/metrics/finetuning/experiment2_consistent_fuzzy_audit.csv


In [40]:

def best_line_match(expected, ocr_text, field):
    expected_norm = normalize_evidence(expected, field)

    if not expected_norm:
        return 0.0, ""

    lines = str(ocr_text).splitlines()

    best_score = 0.0
    best_line = ""

    # Compare individual lines and adjacent line pairs
    candidates = []

    for i, line in enumerate(lines):
        candidates.append(line)

        if i + 1 < len(lines):
            candidates.append(line + " " + lines[i + 1])

    for candidate in candidates:
        candidate_norm = normalize_evidence(candidate, field)

        if not candidate_norm:
            continue

        if expected_norm in candidate_norm:
            return 1.0, candidate

        score = SequenceMatcher(
            None,
            expected_norm,
            candidate_norm
        ).ratio()

        if score > best_score:
            best_score = score
            best_line = candidate

    return best_score, best_line


line_results = []

for _, row in experiment2_cell43.iterrows():

    if row["status"] == "exact_supported":
        continue

    matching_records = [
        r for r in train_records
        if r["document_id"] == row["document_id"]
        and r["ocr_engine"] == row["ocr_engine"]
    ]

    if not matching_records:
        continue

    record = matching_records[0]

    score, matched_line = best_line_match(
        row["expected"],
        record["ocr_text"],
        row["field"]
    )

    line_results.append({
        "document_id": row["document_id"],
        "language": row["language"],
        "ocr_engine": row["ocr_engine"],
        "field": row["field"],
        "expected": row["expected"],
        "whole_text_similarity": row["similarity"],
        "line_similarity": round(score, 4),
        "matched_line": matched_line,
        "supporting_engines": row["supporting_engines"],
        "previous_status": row["status"]
    })


experiment2_line_audit = pd.DataFrame(line_results)

print("Non-exact fields examined:", len(experiment2_line_audit))

print("\nLine-level similarity distribution:")
display(
    experiment2_line_audit["line_similarity"]
    .describe()
    .to_frame("line_similarity")
)

print("\nCandidates with line similarity >= 0.80:")
display(
    experiment2_line_audit[
        experiment2_line_audit["line_similarity"] >= 0.80
    ].sort_values(
        "line_similarity",
        ascending=False
    ).head(40)
)

line_audit_path = os.path.join(
    METRICS_DIR,
    "experiment2_line_level_audit.csv"
)

experiment2_line_audit.to_csv(
    line_audit_path,
    index=False
)

print("\nSaved:", line_audit_path)


Non-exact fields examined: 189

Line-level similarity distribution:


,line_similarity
count,189.000000
mean,0.575071
std,0.277859
min,0.000000
25%,0.421100
50%,0.619000
75%,0.800000
max,0.965500



Candidates with line similarity >= 0.80:


,document_id,language,ocr_engine,field,expected,whole_text_similarity,line_similarity,matched_line,supporting_engines,previous_status
63,hindi_certificate_055,hindi,paddleocr,institution,राष्ट्रीय प्रौद्योगिकी संस्थान,0.9655,0.9655,राष्रीय प्रौद्योगिकी संस्थान,"tesseract, easyocr",cross_ocr_fuzzy_candidate
48,hindi_certificate_051,hindi,paddleocr,institution,राष्ट्रीय प्रौद्योगिकी संस्थान,0.9655,0.9655,राष्रीय प्रौद्योगिकी संस्थान,"tesseract, easyocr",cross_ocr_fuzzy_candidate
146,kannada_certificate_081,kannada,easyocr,institution,ರಾಷ್ಟ್ರೀಯ ತಂತ್ರಜ್ಞಾನ ಸಂಸ್ಥೆ,0.9630,0.9630,ರಾಷ್ಕ್ರೀಯ ತಂತ್ರಜ್ಞಾನ ಸಂಸ್ಥೆ,,fuzzy_only_candidate
10,hindi_invoice_032,hindi,paddleocr,name,अर्जुन वर्मा,0.9565,0.9565,अर्जन वर्मा,"tesseract, easyocr",cross_ocr_fuzzy_candidate
49,hindi_certificate_052,hindi,tesseract,date,21/02/2026,0.9333,0.9333,2/02/2026,"easyocr, paddleocr",cross_ocr_fuzzy_candidate
39,hindi_application_045,hindi,tesseract,date,14/02/2026,0.9333,0.9333,4/02/2026,"easyocr, paddleocr",cross_ocr_fuzzy_candidate
26,hindi_application_041,hindi,paddleocr,address,बेंगलुरु,0.9333,0.9333,बेंगलुर,"tesseract, easyocr",cross_ocr_fuzzy_candidate
23,hindi_application_041,hindi,tesseract,date,10/02/2026,0.9333,0.9333,0/02/2026,"easyocr, paddleocr",cross_ocr_fuzzy_candidate
4,hindi_invoice_031,hindi,easyocr,document_id,INV-0031,0.9231,0.9231,IN//-0031,paddleocr,cross_ocr_fuzzy_candidate
144,kannada_certificate_081,kannada,tesseract,institution,ರಾಷ್ಟ್ರೀಯ ತಂತ್ರಜ್ಞಾನ ಸಂಸ್ಥೆ,0.9200,0.9200,ರಾಷ್ಟ್ರೀಯ ತಂತ್ರಜ್ಞಾನ ಸಂ,,fuzzy_only_candidate



Saved: /content/drive/MyDrive/BharatDocAI/outputs/metrics/finetuning/experiment2_line_level_audit.csv


In [43]:

def classify_experiment2(row):

    # Exact matches were already excluded from Cell 44.
    # Therefore, all rows here are non-exact fields.

    field = row["field"]
    similarity = row["line_similarity"]

    supporting = [
        x.strip()
        for x in str(row["supporting_engines"]).split(",")
        if x.strip()
    ]

    has_cross_ocr_support = len(supporting) >= 1

    if field in ["date", "phone", "amount", "document_id"]:
        threshold = 0.90
    else:
        threshold = 0.93

    if has_cross_ocr_support and similarity >= threshold:
        return "recoverable_candidate"

    return "unavailable_or_uncertain"


experiment2_line_audit["experiment2_status"] = (
    experiment2_line_audit.apply(
        classify_experiment2,
        axis=1
    )
)

print("Experiment 2 proposed evidence categories:")
display(
    experiment2_line_audit[
        "experiment2_status"
    ].value_counts().to_frame("count")
)

print("\nBy language:")
display(
    pd.crosstab(
        experiment2_line_audit["language"],
        experiment2_line_audit["experiment2_status"]
    )
)

print("\nBy field:")
display(
    pd.crosstab(
        experiment2_line_audit["field"],
        experiment2_line_audit["experiment2_status"]
    )
)

print("\nRecoverable candidates:")
display(
    experiment2_line_audit[
        experiment2_line_audit["experiment2_status"]
        == "recoverable_candidate"
    ][[
        "document_id",
        "language",
        "ocr_engine",
        "field",
        "expected",
        "matched_line",
        "line_similarity",
        "supporting_engines"
    ]]
)

cell45_path = os.path.join(
    METRICS_DIR,
    "experiment2_proposed_evidence_categories.csv"
)

experiment2_line_audit.to_csv(
    cell45_path,
    index=False
)

print("\nSaved:", cell45_path)


Experiment 2 proposed evidence categories:


,count
experiment2_status,
unavailable_or_uncertain,178
recoverable_candidate,11



By language:


experiment2_status,recoverable_candidate,unavailable_or_uncertain
language,,
english,0,2
hindi,10,55
kannada,1,121



By field:


experiment2_status,recoverable_candidate,unavailable_or_uncertain
field,,
address,1,14
amount,0,5
date,3,24
document_id,1,67
grade,0,23
institution,2,9
name,1,30
phone,3,6



Recoverable candidates:


,document_id,language,ocr_engine,field,expected,matched_line,line_similarity,supporting_engines
4,hindi_invoice_031,hindi,easyocr,document_id,INV-0031,IN//-0031,0.9231,paddleocr
10,hindi_invoice_032,hindi,paddleocr,name,अर्जुन वर्मा,अर्जन वर्मा,0.9565,"tesseract, easyocr"
23,hindi_application_041,hindi,tesseract,date,10/02/2026,0/02/2026,0.9333,"easyocr, paddleocr"
25,hindi_application_041,hindi,tesseract,phone,9000005617,9000005647,0.9000,"easyocr, paddleocr"
26,hindi_application_041,hindi,paddleocr,address,बेंगलुरु,बेंगलुर,0.9333,"tesseract, easyocr"
39,hindi_application_045,hindi,tesseract,date,14/02/2026,4/02/2026,0.9333,"easyocr, paddleocr"
41,hindi_application_045,hindi,tesseract,phone,9000006165,9000006465,0.9000,"easyocr, paddleocr"
48,hindi_certificate_051,hindi,paddleocr,institution,राष्ट्रीय प्रौद्योगिकी संस्थान,राष्रीय प्रौद्योगिकी संस्थान,0.9655,"tesseract, easyocr"
49,hindi_certificate_052,hindi,tesseract,date,21/02/2026,2/02/2026,0.9333,"easyocr, paddleocr"
63,hindi_certificate_055,hindi,paddleocr,institution,राष्ट्रीय प्रौद्योगिकी संस्थान,राष्रीय प्रौद्योगिकी संस्थान,0.9655,"tesseract, easyocr"



Saved: /content/drive/MyDrive/BharatDocAI/outputs/metrics/finetuning/experiment2_proposed_evidence_categories.csv


In [44]:

# Create a lookup of exact-supported fields from Cell 42
exact_field_keys = set()

for _, row in experiment2_verification.iterrows():
    if row["consistent_exact_match"]:
        exact_field_keys.add((
            row["document_id"],
            row["ocr_engine"],
            row["field"]
        ))

experiment2_training_manifest = []

for record in train_records:
    fields = CATEGORY_SCHEMAS[record["category"]]

    supported_fields = []
    masked_fields = []

    for field in fields:
        key = (
            record["document_id"],
            record["ocr_engine"],
            field
        )

        if key in exact_field_keys:
            supported_fields.append(field)
        else:
            masked_fields.append(field)

    experiment2_training_manifest.append({
        "document_id": record["document_id"],
        "language": record["language"],
        "category": record["category"],
        "ocr_engine": record["ocr_engine"],
        "supported_fields": supported_fields,
        "masked_fields": masked_fields,
        "supported_count": len(supported_fields),
        "masked_count": len(masked_fields)
    })

experiment2_manifest_df = pd.DataFrame(
    experiment2_training_manifest
)

print("Training examples:", len(experiment2_manifest_df))
print(
    "Supported fields:",
    experiment2_manifest_df["supported_count"].sum()
)
print(
    "Masked fields:",
    experiment2_manifest_df["masked_count"].sum()
)

print("\nSupported fields by language:")
display(
    experiment2_manifest_df.groupby("language")[
        ["supported_count", "masked_count"]
    ].sum()
)

print("\nSupported fields by OCR engine:")
display(
    experiment2_manifest_df.groupby("ocr_engine")[
        ["supported_count", "masked_count"]
    ].sum()
)

print("\nExamples with all fields supported:")
print(
    (
        experiment2_manifest_df["masked_count"] == 0
    ).sum()
)

print("\nExamples with at least one masked field:")
print(
    (
        experiment2_manifest_df["masked_count"] > 0
    ).sum()
)

manifest_path = os.path.join(
    OUTPUT_DIR,
    "experiment2_training_manifest.csv"
)

experiment2_manifest_df.to_csv(
    manifest_path,
    index=False
)

print("\nSaved:", manifest_path)


Training examples: 162
Supported fields: 621
Masked fields: 189

Supported fields by language:


,supported_count,masked_count
language,,
english,268,2
hindi,205,65
kannada,148,122



Supported fields by OCR engine:


,supported_count,masked_count
ocr_engine,,
easyocr,234,36
paddleocr,183,87
tesseract,204,66



Examples with all fields supported:
71

Examples with at least one masked field:
91

Saved: /content/drive/MyDrive/BharatDocAI/experiments/finetuning/experiment2_training_manifest.csv


In [47]:

def create_experiment2_example(record):

    fields = CATEGORY_SCHEMAS[record["category"]]

    target = {}
    supported_fields = []
    masked_fields = []

    for field in fields:
        key = (
            record["document_id"],
            record["ocr_engine"],
            field
        )

        expected = str(
            record["ground_truth"].get(field, "")
        ).strip()

        if key in exact_field_keys:
            target[field] = expected
            supported_fields.append(field)
        else:
            target[field] = ""
            masked_fields.append(field)

    target_json = json.dumps(
        target,
        ensure_ascii=False,
        separators=(",", ":")
    )

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT.strip()},
        {
            "role": "user",
            "content": (
                f"Language: {record['language']}\n"
                f"Category: {record['category']}\n"
                f"OCR Text:\n{record['ocr_text']}\n\n"
                f"Extract these fields: {', '.join(fields)}"
            )
        },
        {"role": "assistant", "content": target_json}
    ]

    prompt_text = tokenizer.apply_chat_template(
        messages[:-1],
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    full_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
        enable_thinking=False
    )

    encoded = tokenizer(
        full_text,
        add_special_tokens=False,
        return_offsets_mapping=True
    )

    input_ids = encoded["input_ids"]
    offsets = encoded["offset_mapping"]

    target_start = full_text.rfind(target_json)

    if target_start == -1:
        raise ValueError("Target JSON not found.")

    # Mask everything by default
    labels = [-100] * len(input_ids)

    # Find the full character span of every uncertain property
    masked_character_spans = []

    for field in masked_fields:

        property_text = json.dumps(
            field, ensure_ascii=False
        ) + ':""'

        position = target_json.find(property_text)

        if position == -1:
            raise ValueError(
                f"Could not locate property: {field}"
            )

        start = target_start + position
        end = start + len(property_text)

        masked_character_spans.append((start, end))

    # Supervise assistant JSON tokens except uncertain properties
    for i, (start, end) in enumerate(offsets):

        if end <= start:
            continue

        if start < target_start:
            continue

        overlaps_masked_property = any(
            start < masked_end and end > masked_start
            for masked_start, masked_end in masked_character_spans
        )

        if not overlaps_masked_property:
            labels[i] = input_ids[i]

    if len(input_ids) > MAX_LENGTH:
        raise ValueError(
            f"Sequence too long: {record['document_id']}"
        )

    return {
        "input_ids": input_ids,
        "attention_mask": [1] * len(input_ids),
        "labels": labels,
        "supported_fields": supported_fields,
        "masked_fields": masked_fields,
        "target_json": target_json,
        "document_id": record["document_id"],
        "ocr_engine": record["ocr_engine"],
        "language": record["language"]
    }


experiment2_tokenized_train = [
    create_experiment2_example(record)
    for record in train_records
]

total_supervised = sum(
    sum(label != -100 for label in x["labels"])
    for x in experiment2_tokenized_train
)

total_masked = sum(
    sum(label == -100 for label in x["labels"])
    for x in experiment2_tokenized_train
)

print("Examples:", len(experiment2_tokenized_train))
print("Supervised tokens:", total_supervised)
print("Masked tokens:", total_masked)

print("\nExample with uncertain fields:")

example = next(
    x for x in experiment2_tokenized_train
    if x["masked_fields"]
)

print("Document:", example["document_id"])
print("Target:", example["target_json"])
print("Supported:", example["supported_fields"])
print("Masked:", example["masked_fields"])


Examples: 162
Supervised tokens: 7566
Masked tokens: 32469

Example with uncertain fields:
Document: english_certificate_022
Target: {"name":"Arjun Sharma","date":"22/01/2026","document_id":"CERT-0022","institution":"Government Science College","grade":""}
Supported: ['name', 'date', 'document_id', 'institution']
Masked: ['grade']


In [49]:

assert len(experiment2_tokenized_train) == 162

assert all(
    len(example["input_ids"]) <= MAX_LENGTH
    for example in experiment2_tokenized_train
)

assert all(
    any(label != -100 for label in example["labels"])
    for example in experiment2_tokenized_train
)

assert all(
    len(example["input_ids"]) == len(example["labels"])
    for example in experiment2_tokenized_train
)

total_supervised = sum(
    sum(label != -100 for label in example["labels"])
    for example in experiment2_tokenized_train
)

total_masked = sum(
    sum(label == -100 for label in example["labels"])
    for example in experiment2_tokenized_train
)

print("All integrity checks passed.")
print("Examples:", len(experiment2_tokenized_train))
print("Supervised tokens:", total_supervised)
print("Masked tokens:", total_masked)

print("\nSupported field count:",
      sum(len(x["supported_fields"]) for x in experiment2_tokenized_train))

print("Uncertain field count:",
      sum(len(x["masked_fields"]) for x in experiment2_tokenized_train))

def create_experiment2_example(record):

    fields = CATEGORY_SCHEMAS[record["category"]]

    target = {}
    supported_fields = []
    masked_fields = []

    for field in fields:
        key = (
            record["document_id"],
            record["ocr_engine"],
            field
        )

        expected = str(
            record["ground_truth"].get(field, "")
        ).strip()

        if key in exact_field_keys:
            target[field] = expected
            supported_fields.append(field)
        else:
            target[field] = ""
            masked_fields.append(field)

    target_json = json.dumps(
        target,
        ensure_ascii=False,
        separators=(",", ":")
    )

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT.strip()},
        {
            "role": "user",
            "content": (
                f"Language: {record['language']}\n"
                f"Category: {record['category']}\n"
                f"OCR Text:\n{record['ocr_text']}\n\n"
                f"Extract these fields: {', '.join(fields)}"
            )
        },
        {"role": "assistant", "content": target_json}
    ]

    prompt_text = tokenizer.apply_chat_template(
        messages[:-1],
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    full_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
        enable_thinking=False
    )

    encoded = tokenizer(
        full_text,
        add_special_tokens=False,
        return_offsets_mapping=True
    )

    input_ids = encoded["input_ids"]
    offsets = encoded["offset_mapping"]

    target_start = full_text.rfind(target_json)

    if target_start == -1:
        raise ValueError("Target JSON not found.")

    # Mask everything by default
    labels = [-100] * len(input_ids)

    # Find the full character span of every uncertain property
    masked_character_spans = []

    for field in masked_fields:

        property_text = json.dumps(
            field, ensure_ascii=False
        ) + ':""'

        position = target_json.find(property_text)

        if position == -1:
            raise ValueError(
                f"Could not locate property: {field}"
            )

        start = target_start + position
        end = start + len(property_text)

        masked_character_spans.append((start, end))

    # Supervise assistant JSON tokens except uncertain properties
    for i, (start, end) in enumerate(offsets):

        if end <= start:
            continue

        if start < target_start:
            continue

        overlaps_masked_property = any(
            start < masked_end and end > masked_start
            for masked_start, masked_end in masked_character_spans
        )

        if not overlaps_masked_property:
            labels[i] = input_ids[i]

    if len(input_ids) > MAX_LENGTH:
        raise ValueError(
            f"Sequence too long: {record['document_id']}"
        )

    return {
        "input_ids": input_ids,
        "attention_mask": [1] * len(input_ids),
        "labels": labels,
        "supported_fields": supported_fields,
        "masked_fields": masked_fields,
        "target_json": target_json,
        "document_id": record["document_id"],
        "ocr_engine": record["ocr_engine"],
        "language": record["language"]
    }


experiment2_tokenized_train = [
    create_experiment2_example(record)
    for record in train_records
]

total_supervised = sum(
    sum(label != -100 for label in x["labels"])
    for x in experiment2_tokenized_train
)

total_masked = sum(
    sum(label == -100 for label in x["labels"])
    for x in experiment2_tokenized_train
)

print("Examples:", len(experiment2_tokenized_train))
print("Supervised tokens:", total_supervised)
print("Masked tokens:", total_masked)

print("\nExample with uncertain fields:")

example = next(
    x for x in experiment2_tokenized_train
    if x["masked_fields"]
)

print("Document:", example["document_id"])
print("Target:", example["target_json"])
print("Supported:", example["supported_fields"])
print("Masked:", example["masked_fields"])


All integrity checks passed.
Examples: 162
Supervised tokens: 7566
Masked tokens: 32469

Supported field count: 621
Uncertain field count: 189
Examples: 162
Supervised tokens: 7566
Masked tokens: 32469

Example with uncertain fields:
Document: english_certificate_022
Target: {"name":"Arjun Sharma","date":"22/01/2026","document_id":"CERT-0022","institution":"Government Science College","grade":""}
Supported: ['name', 'date', 'document_id', 'institution']
Masked: ['grade']


In [50]:

def audit_field_token_masking(record, tokenized_example):

    fields = CATEGORY_SCHEMAS[record["category"]]

    target_json = tokenized_example["target_json"]

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT.strip()},
        {
            "role": "user",
            "content": (
                f"Language: {record['language']}\n"
                f"Category: {record['category']}\n"
                f"OCR Text:\n{record['ocr_text']}\n\n"
                f"Extract these fields: {', '.join(fields)}"
            )
        },
        {"role": "assistant", "content": target_json}
    ]

    full_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
        enable_thinking=False
    )

    encoded = tokenizer(
        full_text,
        add_special_tokens=False,
        return_offsets_mapping=True
    )

    offsets = encoded["offset_mapping"]
    labels = tokenized_example["labels"]

    assert len(offsets) == len(labels)

    target_start = full_text.rfind(target_json)
    results = []

    for field in fields:

        key = (
            record["document_id"],
            record["ocr_engine"],
            field
        )

        expected = str(
            record["ground_truth"].get(field, "")
        ).strip()

        is_supported = key in exact_field_keys

        property_text = (
            json.dumps(field, ensure_ascii=False)
            + ":"
            + json.dumps(
                expected if is_supported else "",
                ensure_ascii=False
            )
        )

        position = target_json.find(property_text)

        if position == -1:
            raise ValueError(
                f"Property not found: {field}"
            )

        property_start = target_start + position
        property_end = property_start + len(property_text)

        token_indices = [
            i for i, (start, end) in enumerate(offsets)
            if start < property_end and end > property_start
        ]

        if not token_indices:
            raise ValueError(
                f"No tokens found for {field}"
            )

        supervised = [
            i for i in token_indices
            if labels[i] != -100
        ]

        if is_supported:
            passed = len(supervised) > 0
        else:
            passed = len(supervised) == 0

        results.append({
            "document_id": record["document_id"],
            "field": field,
            "expected_supported": is_supported,
            "property_tokens": len(token_indices),
            "supervised_property_tokens": len(supervised),
            "masked_property_tokens": (
                len(token_indices) - len(supervised)
            ),
            "passed": passed
        })

    return results


audit_results = []

for record, tokenized_example in zip(
    train_records,
    experiment2_tokenized_train
):
    audit_results.extend(
        audit_field_token_masking(
            record,
            tokenized_example
        )
    )

audit_df = pd.DataFrame(audit_results)

print("Total fields audited:", len(audit_df))
print("Passed:", audit_df["passed"].sum())
print("Failed:", (~audit_df["passed"]).sum())

print("\nAudit by support status:")
display(
    audit_df.groupby("expected_supported")["passed"]
    .agg(["count", "sum"])
)

assert audit_df["passed"].all(), (
    "Some fields have incorrect token masking."
)

print("\nAll field-level token masking checks passed!")


Total fields audited: 810
Passed: 810
Failed: 0

Audit by support status:


,count,sum
expected_supported,,
False,189,189
True,621,621



All field-level token masking checks passed!


In [51]:

from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

class Experiment2Dataset(Dataset):

    def __init__(self, examples):
        self.examples = examples

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, index):
        return self.examples[index]


def experiment2_collate_fn(batch):

    input_ids = [
        torch.tensor(x["input_ids"], dtype=torch.long)
        for x in batch
    ]

    attention_masks = [
        torch.tensor(x["attention_mask"], dtype=torch.long)
        for x in batch
    ]

    labels = [
        torch.tensor(x["labels"], dtype=torch.long)
        for x in batch
    ]

    input_ids = pad_sequence(
        input_ids,
        batch_first=True,
        padding_value=tokenizer.pad_token_id
    )

    attention_masks = pad_sequence(
        attention_masks,
        batch_first=True,
        padding_value=0
    )

    labels = pad_sequence(
        labels,
        batch_first=True,
        padding_value=-100
    )

    return {
        "input_ids": input_ids,
        "attention_mask": attention_masks,
        "labels": labels
    }


experiment2_dataset = Experiment2Dataset(
    experiment2_tokenized_train
)

BATCH_SIZE = 2

experiment2_train_loader = DataLoader(
    experiment2_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=experiment2_collate_fn
)

print("Dataset size:", len(experiment2_dataset))
print("Number of batches:", len(experiment2_train_loader))

# Inspect one batch
sample_batch = next(iter(experiment2_train_loader))

for key, value in sample_batch.items():
    print(key, value.shape)

print(
    "\nSupervised tokens in sample batch:",
    (sample_batch["labels"] != -100).sum().item()
)

print(
    "Masked tokens in sample batch:",
    (sample_batch["labels"] == -100).sum().item()
)

# Save the prepared training data for reproducibility
experiment2_dataset_path = os.path.join(
    OUTPUT_DIR,
    "experiment2_tokenized_training.json"
)

with open(experiment2_dataset_path, "w", encoding="utf-8") as f:
    json.dump(
        experiment2_tokenized_train,
        f,
        ensure_ascii=False
    )

print("\nSaved:", experiment2_dataset_path)


Dataset size: 162
Number of batches: 81
input_ids torch.Size([2, 328])
attention_mask torch.Size([2, 328])
labels torch.Size([2, 328])

Supervised tokens in sample batch: 112
Masked tokens in sample batch: 544

Saved: /content/drive/MyDrive/BharatDocAI/experiments/finetuning/experiment2_tokenized_training.json
